# CPBL 主隊勝率預測 — v2：**leak-free, prediction-oriented**（`cpbl_pipeline_v2.ipynb`）

> 這是一個 **複本 (copy) notebook**。v1（`python/cpbl_pipeline.ipynb`）維持鎖定、不被本檔修改。
> v2 沿用 v1 的全部 Colab 慣例（`ROOT = Path.cwd()`、內嵌球場 lookup、rebas 兩季自下載、
> `ts_oof_proba` walk-forward、時間感知切分），STEP-1 ingest 直接逐字沿用 v1 的 cell。

## 這個 v2 是什麼

Lo et al. 2025（*Application of Machine Learning Models for Baseball Outcome
Prediction*, Appl. Sci. 15:7081，同聯盟、同 rebas 資料源）回報 **AUC 0.97–0.98**。
那個數字之所以那麼高，是因為它用**同一場比賽的 box-score 統計**（R、wRC+、wOBA、
OPS、FIP、WHIP、PLOB% 等共 21 打者 ＋ 16 投手變數，全部取自待預測的那一場本身）
去預測該場的 W/L，並以**隨機 5-fold 交叉驗證**評估。**這是 target leakage**：
要先打完那場、累積出那場的 box score，才能算出特徵——預測時根本拿不到。

v2 借用 **論文 Table 1 的 sabermetric 特徵選單**，但把每一個指標都改成**嚴格賽前
(strictly PRE-GAME) 的滾動版本**：只用該隊／該投手在目標場次「之前」的比賽算出來。
v2 的整個重點是把論文的瑕疵**視覺化**（STEP 3 的 `leakage_demo.png`），並在 leak-free
設定下誠實地重做這個預測問題。

### 與 v1 的關係（鎖定的負面結論，v2 不試圖推翻）

v1 已建立：在嚴格 leak-free 特徵 ＋ 時間感知 walk-forward OOF 下，**沒有任何特徵組
能勝過純主場優勢截距**；season-OOF AUC ≈ 0.50–0.53（CI ±0.05），生產贏家 RF
CV-AUC 0.546、holdout 0.640。這是一個**好的負面結果**，方法論才是貢獻。
Vegas MLB 天花板 ≈58.2%、學術 ML 57–59.5%。v2 沿用相同方法論，並把論文 0.97 的
leakage 拍成一張對照圖——**不**改寫 v1 的模型機具、**不**進入調參螺旋。

### 輸出隔離（不可違反）

v2 只寫 `Results/v2/eval/` 與 `Results/v2/figures/`；metrics 檔
`Results/v2/eval/_final_metrics_v2.json`。**絕不**碰 `Results/eval/`、
`Results/figures/`、`python/cpbl_pipeline.ipynb` 或 `reports/00_final_report.md`
——鎖定報告引用那些檔的精確數字。

| Cell | 內容 |
|---|---|
| 2 | 套件 ＋ 路徑 ＋ 內嵌球場 lookup（沿用 v1，外加 `Results/v2/**`）|
| 3 | 抓 rebas 2024+2023（沿用 v1 Cell 3）|
| 4 | 投手資料診斷（沿用 v1 Cell 4）|
| 5 | **STEP 1** ingest → `raw_games.csv`（逐字沿用 v1）|
| 6 | **STEP 3a** leakage demo：same-game wOBA（論文式洩漏）vs pre-game rolling wOBA 的 ROC 對照圖 ← **v2 的核心交付** |
| 7 | **STEP 3b** leak-free sabermetric 特徵工程（論文 Table 1 選單的賽前滾動版）|
| 8 | **STEP 3c** EDA：缺漏審計＋分佈＋相關熱圖＋PCA（誠實判讀，預期弱／無類別分離）|
| 9 | **STEP 3e** pitch-level 賽前特徵（5 區塊，從 PAList 逐球工程；推翻檢定的科學核心）|
| 10 | **STEP 3d** 標準化 ＋ 時間感知切分（絕不隨機）|
| 11 | **STEP 4** 推翻 ablation：(i) saber-only vs (ii) +pitch-level，walk-forward season-OOF ＋ 兩臂各一 stacking ＋ paired bootstrap CI ＋ **>0.70 洩漏 assert 護欄（兩臂都套）** |
| 12 | 結果：印 `_final_metrics_v2.json` ＋ 顯示 v2 圖（含 `pitch_level_ablation.png`）|
| 13 | （選）打包 / 推回 `Results/v2`（只 `git add -f Results/v2`）|

`Runtime → Run all` 由上往下跑完即產出 `Results/v2/figures/leakage_demo.png`
（v2 的論點）、`Results/v2/figures/pitch_level_ablation.png`（推翻檢定）
與 `Results/v2/eval/_final_metrics_v2.json`。

In [ ]:
# === Cell 2 — 套件 + 路徑 + 內嵌球場 lookup（每個 runtime 必跑）===
import subprocess, sys, os
from pathlib import Path
subprocess.run([sys.executable,"-m","pip","install","-q",
                "lightgbm","shap","xgboost"], check=True)
import sklearn, xgboost, lightgbm, shap, pandas, numpy
print("sklearn",sklearn.__version__,"| xgb",xgboost.__version__,
      "| lgb",lightgbm.__version__,"| shap",shap.__version__,
      "| pandas",pandas.__version__)

ROOT = Path.cwd()                       # 所有 step 都以 cwd 為根
for d in ["data/raw/_lookup","data/processed","Results/eval",
          "Results/figures","models"]:
    (ROOT/d).mkdir(parents=True, exist_ok=True)

# step1b 需要球場->經緯度；不 clone repo，故內嵌寫出
LOOKUP_CSV = """stadium_norm,stadium_raw,station_id,station_name,latitude,longitude,note
樂天桃園,樂天桃園棒球場,C0C480,桃園,24.9429,121.2263,exposed; prevailing NW wind in cold months
洲際,臺中市洲際棒球場,72T250,臺中,24.1903,120.6791,bowl shape; moderate wind
天母,臺北市立天母棒球場,466920,臺北,25.1217,121.5324,humid; close to coast
新莊,新北市立新莊棒球場,C0AC60,新莊,25.0460,121.4521,northerly wind off coast
澄清湖,澄清湖棒球場,C0V250,鳳山,22.6738,120.3659,warm humid south
臺南,臺南市立棒球場,467410,臺南,22.9669,120.2138,humid; southerly winds
大巨蛋,臺北大巨蛋,466920,臺北,25.0396,121.5602,indoor; weather is climate context only (is_indoor=1)
其他_嘉義,嘉義市立棒球場,467480,嘉義,23.4833,120.4525,inland hot summer
其他_花蓮,花蓮縣立德興棒球場,466990,花蓮,23.9851,121.6092,coastal east
其他_臺東,臺東棒球村第一棒球場,467660,臺東,22.7596,121.1500,coastal south-east
其他_斗六,斗六棒球場,467480,嘉義,23.7106,120.5460,proxied via 嘉義 station (~25km west)
"""
(ROOT/"data/raw/_lookup/stadium_to_station.csv").write_text(
    LOOKUP_CSV, encoding="utf-8")
print("wrote stadium_to_station.csv (11 venues)")

# ---- v2 OUTPUT ISOLATION: v2 writes ONLY under Results/v2/** -------------
for d in ["Results/v2/eval", "Results/v2/figures"]:
    (ROOT/d).mkdir(parents=True, exist_ok=True)
print("v2 output dirs ready: Results/v2/eval , Results/v2/figures "
      "(v2 NEVER writes Results/eval or Results/figures)")

In [ ]:
# === Cell 3 — 抓 rebas 資料（2024+2023；每個新 runtime 必跑）===
import os, subprocess, glob
USE_2023 = True   # 預設 True：單 2024 已實證無樣本外訊號；2023 把 N 366→~678

B = "https://github.com/rebas-tw/rebas.tw-open-data/releases/download"
urls = [
    f"{B}/v0.1.0-2024/CPBL-2024-OpenData.zip",
    f"{B}/v0.1.0-2024/CPBL-2024-Challenge-OpenData.zip",
    f"{B}/v0.1.0-2024/CPBL-2024-TaiwanSeries-OpenData.zip",
]
if USE_2023:
    urls += [
        f"{B}/v0.1.0-2023.0/CPBL-2023-G1-G150-OpenData.zip",
        f"{B}/v0.1.0-2023.1/CPBL-2023-G151-G300-OpenData.zip",
        f"{B}/v0.1.0-2023.1/CPBL-2023-Challenge-OpenData.zip",
        f"{B}/v0.1.0-2023.1/CPBL-2023-TaiwanSeries-OpenData.zip",
    ]
for u in urls:
    stem = u.split("/")[-1][:-4]
    fn = f"data/raw/{stem}.zip"
    subprocess.run(["wget","-q","-O",fn,u], check=True)
    subprocess.run(["unzip","-o","-q",fn,"-d",f"data/raw/{stem}"], check=True)
# 2024=ASCII (CPBL-2024-OpenData.json) / 2023=中文 (中職2023年-OpenData.json
# 等4種)；只配共同 token 'OpenData'，自動排除 per-game *-G<N>.json
js = sorted(glob.glob("data/raw/**/*OpenData*.json", recursive=True))
print(len(js), "combined JSON:")
for j in js: print("  ", j)
need = 7 if USE_2023 else 3
assert len(js) >= need, f"❌ 預期 >= {need} 個合併檔，只有 {len(js)} — 看上面清單"

In [ ]:
# === Cell 4 — 投手資料診斷（驗證 rebas pitcherBox schema）===
import json, glob, collections, statistics
files = sorted(glob.glob("data/raw/**/*OpenData*.json", recursive=True))
games = []
for f in files: games += json.load(open(f))
print("files:", [f.split("/")[-1] for f in files], "| total games:", len(games))
g = games[0]
print("game keys:", list(g.keys()))
pb = g.get("homePitcherBox") or []
print("homePitcherBox rows:", len(pb))
print("ONE pitcher row:", json.dumps(pb[0], ensure_ascii=False) if pb else "NONE")
bad=0; starts=collections.Counter(); team_sp=collections.defaultdict(set)
seasons=collections.Counter()
for gg in games:
    seasons[str(gg.get("seasonId") or gg.get("season"))[:4]] += 1
    for side,tk in (("homePitcherBox","homeTeam"),("awayPitcherBox","awayTeam")):
        rows=gg.get(side) or []
        s=[r for r in rows if r.get("order")==1]
        if len(s)!=1: bad+=1
        elif s:
            pid=s[0].get("playerId"); starts[pid]+=1; team_sp[gg.get(tk)].add(pid)
sv=list(starts.values())
print("games by season:", dict(seasons))
print("game-sides WITHOUT exactly one order==1:", bad, "/", 2*len(games))
print("distinct starters:", len(sv),
      "| starts/starter mean/median/max:",
      round(statistics.mean(sv),1) if sv else 0,
      statistics.median(sv) if sv else 0, max(sv) if sv else 0)
print("starters per team:", {t:len(s) for t,s in team_sp.items()})

## STEP 1 — ingest（逐字沿用 v1）

下一個 cell 是 v1 `cpbl_pipeline.ipynb` 的 STEP-1 ingest，**逐字沿用**：
一場一列、主／客分別聚合（永不把主＋客相加）、保留每邊 `order == 1` 的先發
投手原始打席線、平手場剔除，輸出 `data/processed/raw_games.csv`。STEP 1 不是
v2 的重點，刻意保持精簡並與 v1 一致（這也保證 ingest 邏輯永不漂移）。

In [ ]:
# === STEP 1 — build raw_games (inlined verbatim from scripts/step1_build_raw_games.py) ===
import json
import hashlib
import re
from datetime import datetime
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
RAW_ROOT = ROOT / "data/raw"
OUT = ROOT / "data/processed"
OUT.mkdir(parents=True, exist_ok=True)
PROV = ROOT / "data/raw/_provenance"
PROV.mkdir(parents=True, exist_ok=True)


def discover_sources(raw_root: Path):
    """Find every combined rebas OpenData JSON under data/raw/ and tag its
    game_type and season from the filename. Returns list[(game_type, season,
    path)] sorted by (season, type-rank) so the time-ordered build is stable.

    NOTE: the 2024 release uses ASCII names (CPBL-2024-OpenData.json); the
    2023 release uses Chinese names (中職2023年-OpenData.json /
    中職2023年下半季-OpenData.json / 中職2023年-季後挑戰賽-OpenData.json /
    中職2023年-台灣大賽-OpenData.json). Match on the shared 'OpenData' token
    only — this still excludes the per-game *-G<N>.json files (no 'OpenData'
    in their names). A 'CPBL-*' prefix would silently drop all 2023 data."""
    type_rank = {"regular": 0, "challenge": 1, "series": 2}
    found = []
    for p in sorted(raw_root.rglob("*OpenData*.json")):
        name = p.name
        if "Challenge" in name or "挑戰" in name:        # 季後挑戰賽
            gtype = "challenge"
        elif "TaiwanSeries" in name or "Series" in name or "台灣大賽" in name:
            gtype = "series"
        else:
            gtype = "regular"
        m = re.search(r"(20\d{2})", name)               # CPBL-2024- / 中職2023年-
        season = int(m.group(1)) if m else 0
        found.append((gtype, season, p))
    found.sort(key=lambda t: (t[1], type_rank.get(t[0], 9)))
    return found


SOURCES = [(g, p) for (g, _s, p) in discover_sources(RAW_ROOT)]
if not SOURCES:
    raise FileNotFoundError(
        f"No *OpenData*.json found under {RAW_ROOT}. "
        "Unzip rebas releases into data/raw/ first."
    )
print("discovered sources:")
for g, p in SOURCES:
    print(f"  [{g:9s}] {p.relative_to(ROOT)}")

# 11 stadiums in source -> 8 normalized levels.
# Bill-James park factor needs N>=20 ideally; the bottom 4 venues all have
# N<10 in 2024 -> collapse to "其他" to avoid overfit on tiny cells.
STADIUM_MAP = {
    "樂天桃園棒球場":     "樂天桃園",
    "臺中市洲際棒球場":   "洲際",
    "臺北市立天母棒球場": "天母",
    "新北市立新莊棒球場": "新莊",
    "澄清湖棒球場":       "澄清湖",
    "臺南市立棒球場":     "臺南",
    "臺北大巨蛋":         "大巨蛋",
    "嘉義市立棒球場":     "其他",
    "花蓮縣立德興棒球場": "其他",
    "臺東棒球村第一棒球場": "其他",
    "斗六棒球場":         "其他",
}
INDOOR_STADIUMS = {"大巨蛋"}

# batter-box stat columns we will re-aggregate per side
BATTER_STAT_KEYS = ["PA", "AB", "R", "H", "RBI", "2B", "3B", "HR",
                    "BB", "IBB", "HBP", "SO", "SH", "SF", "GIDP", "SB", "CS", "E"]

# pitcher-box stat columns (verified against real rebas 2023+2024 rows:
# Cell-4b diagnostic showed every game-side has exactly ONE order==1 row;
# fields IPOuts/NP/BF/H/HR/BB/IBB/HB/SO/R/ER). Staff = sum of all pitchers
# on a side; starter = the order==1 row. step2 turns these into leak-free
# PRIOR-game rolling form (we never use the current game's line for it).
PITCHER_STAT_KEYS = ["IPOuts", "NP", "BF", "H", "HR", "BB",
                     "IBB", "HB", "SO", "R", "ER"]
STARTER_STAT_KEYS = ["IPOuts", "ER", "H", "HR", "BB", "SO", "BF", "NP"]


def sum_inning_scores(arr):
    """Robust sum that skips non-numeric inning entries (e.g. 'X')."""
    s = 0
    for v in arr:
        try:
            s += int(v)
        except (ValueError, TypeError):
            continue
    return s


def aggregate_box(box, prefix):
    out = {f"{prefix}_{k}": 0 for k in BATTER_STAT_KEYS}
    for batter in box:
        for k in BATTER_STAT_KEYS:
            v = batter.get(k, 0) or 0
            try:
                out[f"{prefix}_{k}"] += int(v)
            except (ValueError, TypeError):
                pass
    return out


def aggregate_pitchers(box, prefix):
    """Team pitching-staff totals (sum of every pitcher that appeared).
    'p' prefix keeps these distinct from batter columns (home_pH = hits
    ALLOWED by home pitchers, vs home_H = hits BY home batters)."""
    out = {f"{prefix}_p{k}": 0 for k in PITCHER_STAT_KEYS}
    for pit in box:
        for k in PITCHER_STAT_KEYS:
            v = pit.get(k, 0) or 0
            try:
                out[f"{prefix}_p{k}"] += int(v)
            except (ValueError, TypeError):
                pass
    return out


def extract_starter(box, prefix):
    """The starting pitcher's own line for THIS game = the order==1 row
    (Cell-4b verified: exactly one per side, 0/1356 exceptions). Defensive
    min-by-order so a malformed box still yields the earliest appearance.
    sp_id (playerId) lets step2 roll each starter's OWN prior form."""
    out = {f"{prefix}_sp_id": None}
    out.update({f"{prefix}_sp_{k}": 0 for k in STARTER_STAT_KEYS})
    if not box:
        return out
    sp = min(box, key=lambda r: r.get("order") if r.get("order") is not None else 999)
    out[f"{prefix}_sp_id"] = sp.get("playerId")
    for k in STARTER_STAT_KEYS:
        v = sp.get(k, 0) or 0
        try:
            out[f"{prefix}_sp_{k}"] = int(v)
        except (ValueError, TypeError):
            pass
    return out


def make_game_id(game, game_type, seq_in_source):
    """Build a stable game_id: {date}-{type}-{seq}."""
    date = (game.get("date") or "")[:10].replace("-", "")
    return f"{date}-{game_type[:3].upper()}-{seq_in_source:03d}"


def parse_date(s):
    if not s:
        return pd.NaT
    s = s.replace("/", "-")
    # rebas pattern: "2024-04-04 17:05:00"
    m = re.match(r"^(\d{4})-(\d{1,2})-(\d{1,2})(?:\s+(\d{1,2}):(\d{2})(?::\d{2})?)?", s)
    if not m:
        return pd.NaT
    y, mo, d, hh, mm = m.groups()
    hh = hh or "00"; mm = mm or "00"
    return datetime(int(y), int(mo), int(d), int(hh), int(mm))


# ---------------------------------------------------------------------------
rows = []
provenance = {
    "run_id": datetime.utcnow().isoformat() + "Z",
    "rebas_release_tag": "v0.1.0-2024",
    "sources": {},
}

for game_type, path in SOURCES:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    provenance["sources"][game_type] = {
        "path": str(path.relative_to(ROOT)),
        "n_games": len(data),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }

    for seq, g in enumerate(data, start=1):
        home_scores = g.get("homeScores", [])
        away_scores = g.get("awayScores", [])
        h_total = sum_inning_scores(home_scores)
        a_total = sum_inning_scores(away_scores)

        home_box = g.get("homeBatterBox", []) or []
        away_box = g.get("awayBatterBox", []) or []
        home_pbox = g.get("homePitcherBox", []) or []
        away_pbox = g.get("awayPitcherBox", []) or []

        row = {
            "game_id":    make_game_id(g, game_type, seq),
            "game_type":  game_type,
            "season":     g.get("season"),
            "seasonId":   g.get("seasonId"),
            "seq":        g.get("seq"),
            "datetime":   g.get("date"),
            "stadium_raw": g.get("stadium"),
            "stadium":    STADIUM_MAP.get(g.get("stadium"), "其他"),
            "is_indoor":  int(STADIUM_MAP.get(g.get("stadium"), "其他") in INDOOR_STADIUMS),
            "home_team":  g.get("homeTeam"),
            "away_team":  g.get("awayTeam"),
            "home_team_id": g.get("homeTeamId"),
            "away_team_id": g.get("awayTeamId"),
            "home_innings_played": len(home_scores),
            "away_innings_played": len(away_scores),
            "home_score": h_total,
            "away_score": a_total,
            "total_score": h_total + a_total,
            "is_home_win": int(h_total > a_total),
            "is_tie":      int(h_total == a_total),
            "n_home_batters": len(home_box),
            "n_away_batters": len(away_box),
            "n_home_pitchers": len(home_pbox),
            "n_away_pitchers": len(away_pbox),
        }
        row.update(aggregate_box(home_box, "home"))
        row.update(aggregate_box(away_box, "away"))
        row.update(aggregate_pitchers(home_pbox, "home"))
        row.update(aggregate_pitchers(away_pbox, "away"))
        row.update(extract_starter(home_pbox, "home"))
        row.update(extract_starter(away_pbox, "away"))
        rows.append(row)

df = pd.DataFrame(rows)
df["date"] = df["datetime"].apply(parse_date)
df = df.sort_values(["date", "game_id"]).reset_index(drop=True)

# ---- sanity ----------------------------------------------------------------
print(f"total rows: {len(df)}")
print(f"by game_type: {df['game_type'].value_counts().to_dict()}")
print(f"ties dropped: {df['is_tie'].sum()}")

# 平手場直接丟掉, 我們是 binary classification
df = df.loc[df["is_tie"] == 0].drop(columns=["is_tie"]).reset_index(drop=True)
print(f"after dropping ties: {len(df)}")
print(f"home_win rate: {df['is_home_win'].mean():.3f}")
print(f"stadium normalize roster:")
print(df["stadium"].value_counts().to_string())

# ---- output ----------------------------------------------------------------
out_path = OUT / "raw_games.csv"
df.to_csv(out_path, index=False, encoding="utf-8")
print(f"\nwritten: {out_path}  shape={df.shape}")

provenance["output"] = {
    "path": str(out_path.relative_to(ROOT)),
    "rows": len(df),
    "cols": list(df.columns),
    "sha256": hashlib.sha256(out_path.read_bytes()).hexdigest(),
}
(PROV / "manifest_step1.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"manifest: {PROV / 'manifest_step1.json'}")

## STEP 3a — Leakage demonstration（v2 的核心交付）

> *這張圖就是 v2 的論點。* 它讓論文 0.97 的瑕疵變成**可看見**的東西。

論文用**同一場比賽**的 box-score 算 wOBA 等特徵去預測那場的 W/L。v2 在同一份
資料上做一個**對照實驗**：

1. **Same-game wOBA（蓄意洩漏，論文式設計）**：用待預測那場本身的打者 box
   算 `home_wOBA_samegame − away_wOBA_samegame`，配**隨機 5-fold**（論文用的就是
   shuffle 的 5-fold）的 logistic。預期 **AUC ≈ 0.99**。
2. **Pre-game rolling wOBA（leak-free）**：改用該隊在目標場次「之前」最近 10 場
   的滾動 wOBA 差，**同樣的隨機 5-fold、同樣的 logistic**（蘋果對蘋果）。
   預期 **AUC ≈ 0.53**。

兩條 ROC 畫在同一張軸上 → `Results/v2/figures/leakage_demo.png`。
0.99 vs 0.53 的鴻溝**不是模型變好**，而是「用了不該拿得到的資訊」。這同時
**佐證了 v1 鎖定的負面結論**：賽前真正可得的訊號就只能到 ~0.53。

> 注意：STEP 3a 的 same-game 端**蓄意**使用洩漏特徵 ＋ 隨機 5-fold（重現
> 論文的設定），它**就該**衝到 ≈0.99——這正是要展示的瑕疵。STEP 4 的
> `assert ≤ 0.70` 護欄是針對 **leak-free 的 season-OOF**，不套用在這個
> 蓄意洩漏的對照 demo 上；兩者分屬不同 cell，不是矛盾。

wOBA（FanGraphs 係數）：`1B = H − 2B − 3B − HR`；`uBB = BB − IBB`；
`wOBA = (0.69·uBB + 0.72·HBP + 0.89·1B + 1.27·2B + 1.62·3B + 2.10·HR) /
(AB + BB − IBB + SF + HBP)`，分母 `≤ 0` 時回 NaN。隊伍-場次 wOBA = 先聚合該邊
打者列再套公式。

In [ ]:
# === STEP 3a — LEAKAGE DEMO: same-game vs pre-game wOBA ROC =========
# v2 CENTREPIECE. Makes the paper's same-game-box-score leakage VISUAL.
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve

ROOT = Path.cwd()
FIGV2 = ROOT / "Results/v2/figures"
EVALV2 = ROOT / "Results/v2/eval"
for d in (FIGV2, EVALV2):
    d.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(ROOT / "data/processed/raw_games.csv", parse_dates=["date"])
raw = raw.sort_values(["date", "game_id"]).reset_index(drop=True)
TARGET = "is_home_win"
print(f"raw_games: {len(raw)} games  home-win base={raw[TARGET].mean():.3f}")


def team_woba(side, frame):
    """FanGraphs-style team-game wOBA from a side's aggregated batter
    box (step1 already summed the side's batter rows into <side>_<stat>).
    Denominator <= 0 -> NaN (defensive: empty / malformed box)."""
    H, _2B, _3B, HR = (frame[f"{side}_H"], frame[f"{side}_2B"],
                        frame[f"{side}_3B"], frame[f"{side}_HR"])
    BB, IBB, HBP = frame[f"{side}_BB"], frame[f"{side}_IBB"], frame[f"{side}_HBP"]
    AB, SF = frame[f"{side}_AB"], frame[f"{side}_SF"]
    one_b = H - _2B - _3B - HR
    uBB = BB - IBB
    num = (0.69 * uBB + 0.72 * HBP + 0.89 * one_b
           + 1.27 * _2B + 1.62 * _3B + 2.10 * HR)
    den = AB + BB - IBB + SF + HBP
    return np.where(den > 0, num / den.replace(0, np.nan), np.nan)


# ---- (1) SAME-GAME wOBA (deliberate leak; the paper's design) -----------
raw["home_wOBA_samegame"] = team_woba("home", raw)
raw["away_wOBA_samegame"] = team_woba("away", raw)
raw["diff_wOBA_samegame"] = raw["home_wOBA_samegame"] - raw["away_wOBA_samegame"]

# ---- (2) PRE-GAME rolling wOBA (leak-free counterpart) ------------------
# strictly prior games only: long-form per (team), expanding/rolling
# window of the LAST 10 games BEFORE this one (never the current game).
ROLL = 10
long = []
for _, r in raw.iterrows():
    for side in ("home", "away"):
        long.append({
            "game_id": r["game_id"], "date": r["date"], "side": side,
            "team": r[f"{side}_team"],
            "H": r[f"{side}_H"], "2B": r[f"{side}_2B"], "3B": r[f"{side}_3B"],
            "HR": r[f"{side}_HR"], "BB": r[f"{side}_BB"],
            "IBB": r[f"{side}_IBB"], "HBP": r[f"{side}_HBP"],
            "AB": r[f"{side}_AB"], "SF": r[f"{side}_SF"]})
L = pd.DataFrame(long).sort_values(["team", "date", "game_id"]).reset_index(drop=True)
rec = []
for _team, g in L.groupby("team"):
    g = g.reset_index(drop=True)
    for i in range(len(g)):
        prev = g.iloc[max(0, i - ROLL):i]      # STRICTLY before game i
        if len(prev) < 3:                      # cold start -> NaN
            w = np.nan
        else:
            S = prev[["H", "2B", "3B", "HR", "BB", "IBB", "HBP",
                      "AB", "SF"]].sum()
            one_b = S["H"] - S["2B"] - S["3B"] - S["HR"]
            uBB = S["BB"] - S["IBB"]
            den = S["AB"] + S["BB"] - S["IBB"] + S["SF"] + S["HBP"]
            w = ((0.69 * uBB + 0.72 * S["HBP"] + 0.89 * one_b
                  + 1.27 * S["2B"] + 1.62 * S["3B"] + 2.10 * S["HR"]) / den
                 if den > 0 else np.nan)
        rec.append({"game_id": g.iloc[i]["game_id"],
                    "side": g.iloc[i]["side"], "w": w})
R = pd.DataFrame(rec)
h = R[R.side == "home"][["game_id", "w"]].rename(columns={"w": "home_wOBA_roll10"})
a = R[R.side == "away"][["game_id", "w"]].rename(columns={"w": "away_wOBA_roll10"})
raw = raw.merge(h, on="game_id", how="left").merge(a, on="game_id", how="left")
raw["diff_wOBA_roll10"] = raw["home_wOBA_roll10"] - raw["away_wOBA_roll10"]

# ---- fit both with the SAME simple random 5-fold (the paper's CV) -------
kf = KFold(n_splits=5, shuffle=True, random_state=42)


def cv_roc(col):
    d = raw.dropna(subset=[col]).reset_index(drop=True)
    X = d[[col]].values
    y = d[TARGET].values
    p = cross_val_predict(LogisticRegression(max_iter=2000), X, y,
                           cv=kf, method="predict_proba")[:, 1]
    fpr, tpr, _ = roc_curve(y, p)
    return fpr, tpr, roc_auc_score(y, p), len(d)


fpr_l, tpr_l, auc_leak, n_l = cv_roc("diff_wOBA_samegame")
fpr_p, tpr_p, auc_pre, n_p = cv_roc("diff_wOBA_roll10")
print(f"same-game wOBA (paper-style LEAK)  AUC={auc_leak:.3f}  (n={n_l})")
print(f"pre-game rolling wOBA (leak-free)  AUC={auc_pre:.3f}  (n={n_p})")

fig, ax = plt.subplots(figsize=(7.2, 6.4))
ax.plot(fpr_l, tpr_l, color="#c0392b", lw=2.4,
        label=f"same-game wOBA — paper-style LEAKAGE  (AUC={auc_leak:.3f})")
ax.plot(fpr_p, tpr_p, color="#2471a3", lw=2.4,
        label=f"pre-game rolling wOBA — leak-free  (AUC={auc_pre:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=.5, label="chance (AUC=0.50)")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Leakage demo: same-game box-score (Lo et al. 2025 design)\n"
             "vs strictly pre-game rolling — same data, same model, same CV")
ax.legend(loc="lower right", fontsize=9)
ax.text(0.30, 0.18,
        "The gap is NOT a better model.\n"
        "It is information you cannot have\n"
        "before the game is played.",
        fontsize=9, style="italic",
        bbox=dict(boxstyle="round", fc="#fdf2e9", ec="#c0392b", alpha=.9))
fig.tight_layout()
fig.savefig(FIGV2 / "leakage_demo.png", dpi=130)
plt.close(fig)
print("saved", FIGV2 / "leakage_demo.png")

LEAK_DEMO = {"samegame_wOBA_auc": float(auc_leak),
             "pregame_roll_wOBA_auc": float(auc_pre),
             "n_samegame": int(n_l), "n_pregame": int(n_p),
             "cv": "random KFold(5, shuffle=True) — the paper's design"}
# Persist so STEP 4 reads it back (Colab-robust: if a later STEP-3 cell
# is re-run or fails, STEP 4 still finds the leakage-demo numbers and
# does not lose the model results with a NameError).
import json as _json
(EVALV2 / "_leak_demo.json").write_text(
    _json.dumps(LEAK_DEMO, indent=2, ensure_ascii=False), encoding="utf-8")
print("persisted", EVALV2 / "_leak_demo.json")

## STEP 3b — Leak-free sabermetric feature engineering

借論文 Table 1 的指標選單，但**每一個都改成嚴格賽前的滾動版**：先把比賽按
`(date, seq)` 排序，groupby 隊伍／投手後只取目標場次「之前」的視窗（等同
`shift(1)` 後 rolling）。

**誠實命名**：滾動 wOBA 就叫 `*_wOBA_roll10`，**不**叫 `wRC+`——v2 沒有做
park／league 調整，把它叫 wRC+ 會是另一種誤導。論文 0.97 的核心驅動特徵
（論文 SHAP 指認的 wRC+、PLOB%）正是 same-game 量；v2 不重蹈。

- **打線（last 10 場滾動，per team）**：`*_wOBA_roll10`、`*_OPS_roll10`、
  `*_OBP_roll10`、`*_runs_for_roll10`、`*_runs_against_roll10`，及對應
  `diff_*`。
- **先發投手（該投手自身 last 5 starts，主客合併，因投球能力與場地無關）**：
  `*_sp_FIP_l5`、`*_sp_WHIP_l5`、`*_sp_KBB_l5`，及 `diff_*`。
  FIP `= (13·HR + 3·(BB+HB) − 2·SO)/(IPOuts/3) + 3.1`；
  WHIP `= (BB+H)/(IPOuts/3)`；K/BB `= SO/max(BB,1)`。
- **球隊戰力**：Pythagenpat 期望勝率（由滾動 runs for/against，指數
  自 RS+RA 動態決定）`*_pythag` ＋ `diff_pythag`；主／客休息天數
  `*_rest_days` ＋ `diff_rest`。

**冷啟動規則（文件化）**：某隊／某投手在目標場次前**少於 3 場**先前觀測時，
該滾動特徵設為 `NaN`；建模 pipeline 以 **median imputation** 回退（＝聯盟
平均水準）。此規則同時套用在打線、投手、Pythag。所有特徵 NaN-tolerant。

In [ ]:
# === STEP 3b — leak-free sabermetric feature engineering ===========
# Honest names: rolling wOBA is *_wOBA_roll10 (NOT "wRC+": no park/league
# adjustment is applied). STRICTLY pre-game: groupby team/pitcher, take
# only games chronologically BEFORE the target game. <3 prior obs -> NaN
# (median-imputed in STEP 4's pipeline = league-average fallback).
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data/processed/raw_games.csv", parse_dates=["date"])
df = df.sort_values(["date", "seq", "game_id"]).reset_index(drop=True)
TARGET = "is_home_win"
ROLL_BAT = 10            # batter rolling window (last N team games)
ROLL_SP = 5              # starter rolling window (last N of that pitcher's starts)
MIN_PRIOR = 3            # < this many prior obs -> NaN (cold start)
PYTHAG_WIN = 10          # rolling window for Pythagenpat runs for/against

LEAKFREE_SABER = []      # the leak-free sabermetric feature block (diffs)


def _woba(S):
    one_b = S["H"] - S["2B"] - S["3B"] - S["HR"]
    uBB = S["BB"] - S["IBB"]
    den = S["AB"] + S["BB"] - S["IBB"] + S["SF"] + S["HBP"]
    if den <= 0:
        return np.nan
    return ((0.69 * uBB + 0.72 * S["HBP"] + 0.89 * one_b + 1.27 * S["2B"]
             + 1.62 * S["3B"] + 2.10 * S["HR"]) / den)


def _obp_ops(S):
    one_b = S["H"] - S["2B"] - S["3B"] - S["HR"]
    ab_den = S["AB"]
    obp_den = S["AB"] + S["BB"] + S["HBP"] + S["SF"]
    if not ab_den or not obp_den:
        return np.nan, np.nan
    obp = (S["H"] + S["BB"] + S["HBP"]) / obp_den
    slg = (one_b + 2 * S["2B"] + 3 * S["3B"] + 4 * S["HR"]) / ab_den
    return obp, obp + slg


# ---- A. team batting: wOBA / OPS / OBP / runs_for / runs_against -------
rows = []
for _, r in df.iterrows():
    for side in ("home", "away"):
        other = "away" if side == "home" else "home"
        rows.append({
            "game_id": r["game_id"], "date": r["date"], "seq": r["seq"],
            "side": side, "team": r[f"{side}_team"],
            "H": r[f"{side}_H"], "2B": r[f"{side}_2B"], "3B": r[f"{side}_3B"],
            "HR": r[f"{side}_HR"], "BB": r[f"{side}_BB"],
            "IBB": r[f"{side}_IBB"], "HBP": r[f"{side}_HBP"],
            "AB": r[f"{side}_AB"], "SF": r[f"{side}_SF"],
            "rs": r[f"{side}_score"], "ra": r[f"{other}_score"]})
L = (pd.DataFrame(rows)
     .sort_values(["team", "date", "seq", "game_id"]).reset_index(drop=True))
bat_cols = ["H", "2B", "3B", "HR", "BB", "IBB", "HBP", "AB", "SF", "rs", "ra"]
feat = []
for _team, g in L.groupby("team"):
    g = g.reset_index(drop=True)
    for i in range(len(g)):
        prev = g.iloc[max(0, i - ROLL_BAT):i]      # strictly prior
        row = {"game_id": g.iloc[i]["game_id"], "side": g.iloc[i]["side"]}
        if len(prev) < MIN_PRIOR:
            for c in ["wOBA", "OPS", "OBP", "runs_for", "runs_against"]:
                row[c] = np.nan
        else:
            S = prev[bat_cols].sum()
            obp, ops = _obp_ops(S)
            row["wOBA"] = _woba(S)
            row["OPS"] = ops
            row["OBP"] = obp
            row["runs_for"] = S["rs"] / len(prev)
            row["runs_against"] = S["ra"] / len(prev)
        feat.append(row)
F = pd.DataFrame(feat)
for col in ["wOBA", "OPS", "OBP", "runs_for", "runs_against"]:
    hh = F[F.side == "home"][["game_id", col]].rename(
        columns={col: f"home_{col}_roll{ROLL_BAT}"})
    aa = F[F.side == "away"][["game_id", col]].rename(
        columns={col: f"away_{col}_roll{ROLL_BAT}"})
    df = df.merge(hh, on="game_id", how="left").merge(aa, on="game_id", how="left")
    dname = f"diff_{col}_roll{ROLL_BAT}"
    df[dname] = (df[f"home_{col}_roll{ROLL_BAT}"]
                 - df[f"away_{col}_roll{ROLL_BAT}"])
    LEAKFREE_SABER.append(dname)

# ---- B. starting-pitcher own last-5 form: FIP / WHIP / K/BB ----------
rows = []
for _, r in df.iterrows():
    for side in ("home", "away"):
        spid = r.get(f"{side}_sp_id")
        if spid is None or (isinstance(spid, float) and pd.isna(spid)):
            continue
        rows.append({
            "game_id": r["game_id"], "date": r["date"], "seq": r["seq"],
            "side": side, "sp_id": spid,
            "IPOuts": r[f"{side}_sp_IPOuts"], "H": r[f"{side}_sp_H"],
            "HR": r[f"{side}_sp_HR"], "BB": r[f"{side}_sp_BB"],
            "SO": r[f"{side}_sp_SO"],
            "HB": r.get(f"{side}_sp_HB", 0)})    # HBP-against (may be absent)
Lsp = (pd.DataFrame(rows)
       .sort_values(["sp_id", "date", "seq", "game_id"]).reset_index(drop=True))
feat = []
for _spid, g in Lsp.groupby("sp_id"):
    g = g.reset_index(drop=True)
    for i in range(len(g)):
        prev = g.iloc[max(0, i - ROLL_SP):i]       # this pitcher's prior starts
        row = {"game_id": g.iloc[i]["game_id"], "side": g.iloc[i]["side"]}
        if len(prev) < MIN_PRIOR:
            row["sp_FIP"] = row["sp_WHIP"] = row["sp_KBB"] = np.nan
        else:
            S = prev[["IPOuts", "H", "HR", "BB", "SO", "HB"]].sum()
            ip = S["IPOuts"] / 3.0
            row["sp_FIP"] = (((13 * S["HR"] + 3 * (S["BB"] + S["HB"])
                               - 2 * S["SO"]) / ip + 3.1)
                             if ip > 0 else np.nan)
            row["sp_WHIP"] = ((S["BB"] + S["H"]) / ip) if ip > 0 else np.nan
            row["sp_KBB"] = S["SO"] / max(S["BB"], 1)
        feat.append(row)
Fsp = pd.DataFrame(feat)
for col in ["sp_FIP", "sp_WHIP", "sp_KBB"]:
    hh = Fsp[Fsp.side == "home"][["game_id", col]].rename(
        columns={col: f"home_{col}_l{ROLL_SP}"})
    aa = Fsp[Fsp.side == "away"][["game_id", col]].rename(
        columns={col: f"away_{col}_l{ROLL_SP}"})
    df = df.merge(hh, on="game_id", how="left").merge(aa, on="game_id", how="left")
    dname = f"diff_{col}_l{ROLL_SP}"
    df[dname] = df[f"home_{col}_l{ROLL_SP}"] - df[f"away_{col}_l{ROLL_SP}"]
    LEAKFREE_SABER.append(dname)

# ---- C. Pythagenpat expected win% from rolling runs for/against ------
# exponent x = ((RS+RA)/G) ** 0.287  (Pythagenpat; data-driven, not fixed).
rows = []
for _, r in df.iterrows():
    for side in ("home", "away"):
        other = "away" if side == "home" else "home"
        rows.append({"game_id": r["game_id"], "date": r["date"],
                     "seq": r["seq"], "side": side, "team": r[f"{side}_team"],
                     "rs": r[f"{side}_score"], "ra": r[f"{other}_score"]})
Lp = (pd.DataFrame(rows)
      .sort_values(["team", "date", "seq", "game_id"]).reset_index(drop=True))
feat = []
for _team, g in Lp.groupby("team"):
    g = g.reset_index(drop=True)
    for i in range(len(g)):
        prev = g.iloc[max(0, i - PYTHAG_WIN):i]
        gid = g.iloc[i]["game_id"]
        side = g.iloc[i]["side"]
        if len(prev) < MIN_PRIOR:
            feat.append({"game_id": gid, "side": side, "pythag": np.nan})
            continue
        rs, ra = prev["rs"].sum(), prev["ra"].sum()
        if rs + ra == 0:
            feat.append({"game_id": gid, "side": side, "pythag": 0.5})
        else:
            x = ((rs + ra) / len(prev)) ** 0.287
            feat.append({"game_id": gid, "side": side,
                         "pythag": rs ** x / (rs ** x + ra ** x)})
Fp = pd.DataFrame(feat)
hh = Fp[Fp.side == "home"][["game_id", "pythag"]].rename(
    columns={"pythag": "home_pythag"})
aa = Fp[Fp.side == "away"][["game_id", "pythag"]].rename(
    columns={"pythag": "away_pythag"})
df = df.merge(hh, on="game_id", how="left").merge(aa, on="game_id", how="left")
df["diff_pythag"] = df["home_pythag"] - df["away_pythag"]
LEAKFREE_SABER.append("diff_pythag")

# ---- D. rest days (home / away), cap 5 -------------------------------
rows = []
for _, r in df.iterrows():
    for side in ("home", "away"):
        rows.append({"game_id": r["game_id"], "date": r["date"],
                     "seq": r["seq"], "side": side, "team": r[f"{side}_team"]})
Lr = (pd.DataFrame(rows)
      .sort_values(["team", "date", "seq", "game_id"]).reset_index(drop=True))
Lr["prev_date"] = Lr.groupby("team")["date"].shift(1)
Lr["rest"] = (Lr["date"] - Lr["prev_date"]).dt.days.clip(upper=5)
wide = Lr.pivot_table(index="game_id", columns="side", values="rest").reset_index()
wide = wide.rename(columns={"home": "home_rest_days", "away": "away_rest_days"})
df = df.merge(wide, on="game_id", how="left")
for c in ["home_rest_days", "away_rest_days"]:
    df[c] = df[c].fillna(df[c].median())
df["diff_rest"] = df["home_rest_days"] - df["away_rest_days"]
LEAKFREE_SABER.append("diff_rest")

# ---- cold-start coverage + persist -----------------------------------
df["features_complete"] = (
    df[LEAKFREE_SABER].isna().sum(axis=1) == 0).astype(int)
print(f"leak-free sabermetric features ({len(LEAKFREE_SABER)}): "
      f"{LEAKFREE_SABER}")
print(f"rows with all features present (post warm-up): "
      f"{int(df['features_complete'].sum())} / {len(df)}")
# HONEST-NAMING GUARD (executable): no rolling feature is called wRC+.
assert not any("wRC" in c or "wrcplus" in c.lower() for c in df.columns), (
    "honest-naming violation: a rolling column is named wRC+ "
    "(no park/league adjustment is applied — must stay *_wOBA_roll*)")
V2_CSV = ROOT / "data/processed/model_ready_v2.csv"
df.to_csv(V2_CSV, index=False, encoding="utf-8")
print("written", V2_CSV, "shape", df.shape)

# Persist the EXPLICIT saber feature list (STEP 3e appends its pitch
# list; STEP 4 reads both). Explicit lists kill any chance of the wrong
# column sneaking into the wrong ablation arm via prefix detection.
import json as _json
from pathlib import Path as _P
_fl_path = _P.cwd() / "Results/v2/eval/_feature_lists.json"
_fl_path.parent.mkdir(parents=True, exist_ok=True)
_fl_path.write_text(_json.dumps(
    {"saber": LEAKFREE_SABER, "pitch_level": []},
    indent=2, ensure_ascii=False), encoding="utf-8")
print("persisted feature list (saber):", _fl_path)

## STEP 3c — EDA（誠實判讀，預期弱／無類別分離）

四項純探索分析，**不**用來提升 AUC——它們從非監督角度獨立印證
「賽前訊號很弱」這個 v1 鎖定的結論：

1. **缺漏審計表**：每個 leak-free 特徵的缺漏率（冷啟動造成，已記錄規則）。
2. **特徵分佈 hist 格**（`Results/v2/figures/eda_distributions.png`）。
3. **相關熱圖**（`Results/v2/figures/eda_corr.png`）：leak-free 特徵彼此的
   Pearson 相關，看是否有冗餘 / 共線。
4. **PCA**（`Results/v2/figures/eda_pca.png`）：標準化 leak-free sabermetric
   區塊的 scree ＋ PC1/PC2 散點依 `is_home_win` 上色——預期兩類**重疊、
   無可分性**，這正是 finding 本身。

In [ ]:
# === STEP 3c — EDA: missingness / distributions / corr / PCA =======
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

ROOT = Path.cwd()
FIGV2 = ROOT / "Results/v2/figures"
FIGV2.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(ROOT / "data/processed/model_ready_v2.csv", parse_dates=["date"])
TARGET = "is_home_win"
SABER = [c for c in df.columns
         if c.startswith("diff_") and c not in ("diff_score",)]
print("leak-free sabermetric block:", SABER)

# ---- (1) missingness audit -------------------------------------------
miss = (df[SABER].isna().mean().mul(100).round(2)
        .sort_values(ascending=False)
        .rename("missing_%").to_frame())
miss["n_present"] = df[SABER].notna().sum().reindex(miss.index).values
print("\nMISSINGNESS AUDIT (cold-start <3 prior obs -> NaN -> "
      "median-imputed in STEP 4):")
print(miss.to_string())

# ---- (2) feature distributions (hist grid) ---------------------------
nb = len(SABER)
ncol = 4
nrow = int(np.ceil(nb / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3 * nrow))
axes = np.atleast_1d(axes).ravel()
for j, c in enumerate(SABER):
    axes[j].hist(df[c].dropna(), bins=30, color="#48a", alpha=.85)
    axes[j].set_title(c, fontsize=9)
for k in range(nb, len(axes)):
    axes[k].axis("off")
fig.suptitle("Leak-free sabermetric feature distributions", y=1.002)
fig.tight_layout()
fig.savefig(FIGV2 / "eda_distributions.png", dpi=120,
            bbox_inches="tight")
plt.close(fig)
print("saved", FIGV2 / "eda_distributions.png")

# ---- (3) correlation heatmap -----------------------------------------
C = df[SABER].corr()
fig, ax = plt.subplots(figsize=(1.1 * len(SABER) + 2, 1.0 * len(SABER) + 2))
im = ax.imshow(C.values, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(SABER)))
ax.set_xticklabels(SABER, rotation=90, fontsize=8)
ax.set_yticks(range(len(SABER)))
ax.set_yticklabels(SABER, fontsize=8)
for ii in range(len(SABER)):
    for jj in range(len(SABER)):
        ax.text(jj, ii, f"{C.values[ii, jj]:.2f}", ha="center",
                va="center", fontsize=7,
                color="white" if abs(C.values[ii, jj]) > .6 else "black")
fig.colorbar(im, fraction=.046, pad=.04)
ax.set_title("Leak-free feature correlation (Pearson)")
fig.tight_layout()
fig.savefig(FIGV2 / "eda_corr.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("saved", FIGV2 / "eda_corr.png")

# ---- (4) PCA on standardised leak-free block -------------------------
G = df[SABER].replace([np.inf, -np.inf], np.nan)
G = G.fillna(G.median())
Xs = StandardScaler().fit_transform(G)
pca = PCA().fit(Xs)
evr = pca.explained_variance_ratio_
Z = pca.transform(Xs)[:, :2]
y = df[TARGET].values
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].bar(range(1, len(evr) + 1), evr, color="#48a")
ax[0].plot(range(1, len(evr) + 1), np.cumsum(evr), "o-r")
ax[0].set_title(f"PCA scree (PC1+PC2 = {evr[:2].sum():.0%})")
ax[0].set_xlabel("PC")
ax[0].set_ylabel("explained variance ratio")
for v, c, l in [(1, "#2471a3", "home win"), (0, "#d98", "home loss")]:
    m = y == v
    ax[1].scatter(Z[m, 0], Z[m, 1], s=10, alpha=.4, color=c, label=l)
ax[1].set_xlabel("PC1")
ax[1].set_ylabel("PC2")
ax[1].set_title("Leak-free PC space — no outcome separation (expected)")
ax[1].legend()
fig.tight_layout()
fig.savefig(FIGV2 / "eda_pca.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("saved", FIGV2 / "eda_pca.png")
print("\nEDA READING (honest): the missingness is purely cold-start "
      "(early-season games, <3 prior obs); distributions are unimodal "
      "and roughly symmetric; the correlation block shows the expected "
      "wOBA/OPS/OBP collinearity but no feature dominates; the PCA "
      "scatter shows home-win and home-loss clouds almost completely "
      "OVERLAPPING with a near-flat scree. Weak/absent class separation "
      "IS the finding — it independently corroborates v1's locked "
      "negative result before any model is fit.")

## STEP 3e — Pitch-level pre-game features（推翻檢定的科學核心）

> 這一步**新增**的特徵，全部從 `homePAList` / `awayPAList`
> 逐球(pitch-by-pitch)資料工程出來，**每一個都是嚴格賽前 (strictly
> PRE-GAME) 的滾動／展開聚合**：把比賽按 `(date, seq)` 排序，groupby
> 隊伍／投手後**只取目標場次之前**的視窗（等同 `shift(1)` 後 rolling）。
> 目的：檢定「逐球層級的賽前訊號」能否推翻 v1 鎖定的負面結論
> （sabermetric-only ≈ 0.53 season-OOF AUC）。**不**為了衝數字而放寬任何
> leak-free 規則。

STEP-1 的 `raw_games.csv` 只聚合 box-score（沒有用到 PAList）；STEP 3e
**自己重新解析**那批 `data/raw/**/*OpenData*.json`，並用與 STEP-1
**逐字相同**的 `discover_sources` 排序 ＋ `make_game_id`
（`{YYYYMMDD}-{TYPE[:3]}-{enumerate_seq:03d}`）重建 `game_id`，所以新特徵
能正確 merge 進**同一個** `model_ready_v2.csv`（STEP 3d 切分、STEP 4 消費
的就是它）——是整合，不是死掛件。

### 五個 leak-free 區塊（每個 → home/away → 再 `diff_*`）

1. **先發 stuff/command**（該場先發投手自身 last 5 starts 滾動）：平均球速、
   球速趨勢斜率、球種 Shannon entropy、zone%、揮空好球率(swstr%)、
   首球好球率(fps%)。
2. **先發 platoon split**（同先發 last 5 starts，依對方打線左右打比例加權的
   K% / BB%；小樣本 → 朝**凍結校準窗**的聯盟均值收縮 shrink）。
3. **打線接觸品質＋選球**（該隊 last 10 場滾動）：強擊球率、平飛球率、
   GB/FB、追打率(chase%)、好球帶內接觸率、K%、BB%。
4. **牛棚 pitch quality**（該隊 **後援**投手 = 非該場先發者，last 30 場
   滾動）：平均球速、揮空好球率、上壘 proxy。與先發區塊區隔。
5. **Form 化 WPA/RE24**（per-batter 嚴格賽前 expanding 平均，再聚到隊-場）。
   **命名即洩漏護欄**：欄位名強制以 `_prior_s2d` 結尾
   （`team_WPA_prior_s2d`、`team_RE24_prior_s2d`），**絕不**用裸 `WPA`
   / `RE24` / `homeWE`——目標場次的 within-game 狀態量零使用。

### 經驗式好球帶推導（modeling decision，已文件化）

逐球的 `coordX/coordY` 沒有官方好球帶座標。本檔的規則：取所有
**「被主審判定為看球好球」(pitchCode == 'S'，看打、未揮、未進場)** 的球，
其 `coordX` 與 `coordY` 各自的 **第 10–90 百分位**框出中央矩形 → 經驗
好球帶。`zone%`、`chase%`、`zone-contact%` 都依這個盒界判定。

**這個盒界、以及 platoon 的聯盟收縮目標，是兩個 PRE-REGISTERED
MODELLING CONSTANT**：只從一個**凍結校準窗 (frozen calibration
window)** 推導**一次**、整個 walk-forward 全程固定不變——Colab 上
校準窗 = **整個 2023 季**（嚴格早於任何 2024 預測目標）；只有 2023 的
本機 smoke test 退回**最早 30% 場次**。如此這兩個常數**依建構就與任何
單一目標場次無關**，不是洩漏。

### 洩漏規則（已烤進程式碼＋本說明，不可協商）
- `WPA, RE24, homeWE, RE`：within-game 狀態。只能當 per-player 賽前滾動
  平均、且欄名 `*_prior_s2d`；目標場次的值零使用。
- `result, scored, endOuts, endBases, RBI`：PA 結果。只進賽前 per-player
  聚合，絕不進目標場次自己那一列。
- `coordX/coordY`：只用於滾動 zone%/chase%；絕不把目標場次同場逐球序列
  當特徵。
- 最細匹配粒度 = 隊 vs 先發（含朝聯盟均值收縮）；**不**做打者 vs 特定投手
  matchup（樣本過小、雜訊）。
- 冷啟動（某實體賽前 < 3 筆觀測）：該滾動值設 `NaN`，由 STEP 4 既有的
  `SimpleImputer(median)` 回退（＝聯盟平均水準）。

> **重要設計決定**：STEP 3e **不更新** `features_complete`。那個旗標是
> sabermetric 的冷啟動遮罩；pitch-level 的 NaN 一律走 STEP 4 的中位數
> 補值。理由：**ablation 的兩臂必須跑在完全相同的 N 上**，否則 (i) 與
> (ii) 不是蘋果對蘋果。本說明把這點明文記錄。

> **本機 smoke test 已實證**（2023 全季 299 場、22.9k PA、86.7k 逐球）：
> 8 個抽樣目標場 × 5 區塊的**逐區塊洩漏探針**全數 `d < 1e-9`——丟掉所有
> `date >= G.date` 的比賽不會改變 G 的任何 STEP-3e 特徵列。

In [ ]:
# === STEP 3e — pitch-level PRE-GAME features (overturn test) =======
# Re-parses data/raw/**/*OpenData*.json (STEP-1 only used box-score, not
# PAList) with the SAME discover_sources sort + make_game_id as STEP-1 so
# the new diff_* columns merge into the SAME model_ready_v2.csv that
# STEP 3d splits and STEP 4 consumes. Every feature is a STRICTLY
# pre-game rolling/expanding aggregate. Strike zone + platoon league
# shrink target are PRE-REGISTERED constants frozen from a calibration
# window (Colab: all 2023; 2023-only: earliest 30%) -> by construction
# independent of any target game. features_complete is NOT updated (the
# ablation's two arms must run on the same N).
import json
import math
import re
import glob
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
EVALV2 = ROOT / "Results/v2/eval"
EVALV2.mkdir(parents=True, exist_ok=True)

ROLL_SP_PL = 5            # starter own last-N starts
ROLL_BAT_PL = 10          # lineup last-N team games
ROLL_PEN_PL = 30          # bullpen last-N team games
MIN_PRIOR_PL = 3          # < this many prior obs -> NaN (cold start)
VEL_LO, VEL_HI = 60.0, 175.0
SWING_CODES = {"SW", "F", "FT", "H", "BUNT", "TRY_BUNT"}
WHIFF_CODES = {"SW", "TRY_BUNT"}          # swing-and-miss
CALLED_STRIKE = "S"                       # clean looking-strike code
LEAGUE_SHRINK_K = 50.0                    # pseudo-PA for platoon shrink
CALIB_END_DATE = "2024-01-01"             # Colab: all 2023 = calib window
CALIB_FRAC = 0.30                         # 2023-only fallback


# ---- discover_sources / make_game_id : VERBATIM from v1 STEP-1 -------
def _discover_sources(raw_root):
    type_rank = {"regular": 0, "challenge": 1, "series": 2}
    found = []
    for p in sorted(Path(raw_root).rglob("*OpenData*.json")):
        name = p.name
        if "Challenge" in name or "挑戰" in name:
            gtype = "challenge"
        elif ("TaiwanSeries" in name or "Series" in name
              or "台灣大賽" in name):
            gtype = "series"
        else:
            gtype = "regular"
        m = re.search(r"(20\d{2})", name)
        season = int(m.group(1)) if m else 0
        found.append((gtype, season, p))
    found.sort(key=lambda t: (t[1], type_rank.get(t[0], 9)))
    return found


def _make_game_id(game, game_type, seq_in_source):
    date = (game.get("date") or "")[:10].replace("-", "")
    return f"{date}-{game_type[:3].upper()}-{seq_in_source:03d}"


def _pdate(s):
    if not s:
        return pd.NaT
    s = s.replace("/", "-")
    m = re.match(r"^(\d{4})-(\d{1,2})-(\d{1,2})"
                 r"(?:\s+(\d{1,2}):(\d{2}))?", s)
    if not m:
        return pd.NaT
    y, mo, d, hh, mm = m.groups()
    return pd.Timestamp(int(y), int(mo), int(d), int(hh or 0),
                        int(mm or 0))


def _f(v):
    try:
        x = float(v)
        return x if math.isfinite(x) else np.nan
    except (TypeError, ValueError):
        return np.nan


def _build_long(sources):
    games, pas, pitches = [], [], []
    for gtype, _s, path in sources:
        with open(path, encoding="utf-8") as fh:
            data = json.load(fh)
        for seq, g in enumerate(data, start=1):
            gid = _make_game_id(g, gtype, seq)
            gdate = _pdate(g.get("date"))
            sseq = g.get("seq")
            h = sum(int(x) for x in g.get("homeScores", [])
                    if str(x).lstrip("-").isdigit())
            a = sum(int(x) for x in g.get("awayScores", [])
                    if str(x).lstrip("-").isdigit())
            if h == a:                       # ties dropped (mirror STEP-1)
                continue
            sp_name = {}
            for side in ("home", "away"):
                box = g.get(f"{side}PitcherBox", []) or []
                sp = (min(box, key=lambda r: r.get("order")
                          if r.get("order") is not None else 999)
                      if box else None)
                sp_name[side] = sp.get("playerName") if sp else None
            games.append({"game_id": gid, "date": gdate, "g_seq": sseq,
                          "home_team": g.get("homeTeam"),
                          "away_team": g.get("awayTeam"),
                          "home_sp_name": sp_name["home"],
                          "away_sp_name": sp_name["away"]})
            for side in ("home", "away"):
                bteam = g.get(f"{side}Team")
                for pa in g.get(f"{side}PAList", []) or []:
                    pas.append({
                        "game_id": gid, "date": gdate, "g_seq": sseq,
                        "bat_side": side, "bat_team": bteam,
                        "batterName": pa.get("batterName"),
                        "batterHand": pa.get("batterHand"),
                        "pitcherName": pa.get("pitcherName"),
                        "result": pa.get("result"),
                        "hardness": pa.get("hardness"),
                        "trajectory": pa.get("trajectory"),
                        "WPA": _f(pa.get("WPA")),
                        "RE24": _f(pa.get("RE24"))})
                    for ev in pa.get("events", []) or []:
                        if ev.get("type") != "PITCH":
                            continue
                        v = _f(ev.get("velocity"))
                        if not (VEL_LO <= v <= VEL_HI):
                            v = np.nan
                        pitches.append({
                            "game_id": gid, "date": gdate,
                            "g_seq": sseq, "bat_side": side,
                            "bat_team": bteam,
                            "pitcherName": ev.get("pitcherName"),
                            "pitchType": ev.get("pitchType"),
                            "pitchCode": ev.get("pitchCode"),
                            "velocity": v,
                            "coordX": _f(ev.get("coordX")),
                            "coordY": _f(ev.get("coordY")),
                            "isStrike": bool(ev.get("isStrike")),
                            "inPlay": bool(ev.get("inPlay"))})
    G = (pd.DataFrame(games)
         .sort_values(["date", "game_id"]).reset_index(drop=True))
    return G, pd.DataFrame(pas), pd.DataFrame(pitches)


def _calib_cutoff(G):
    """End of the frozen calibration window. Colab: 2024-01-01 (all
    2023). 2023-only data: earliest CALIB_FRAC of games."""
    cut = pd.Timestamp(CALIB_END_DATE)
    if (G["date"] < cut).sum() >= 30:
        return cut
    order = G.sort_values(["date", "game_id"]).reset_index(drop=True)
    k = max(1, int(len(order) * CALIB_FRAC))
    return order.loc[min(k, len(order) - 1), "date"]


def _strike_zone(PI, cutoff):
    """p10..p90 of coordX AND coordY over CALLED strikes (pitchCode=='S'
    : umpire-judged in-zone, no-swing) in the frozen calib window."""
    cs = PI[(PI["pitchCode"] == CALLED_STRIKE)
            & PI["coordX"].notna() & PI["coordY"].notna()
            & (PI["date"] < cutoff)]
    x, y = cs["coordX"].to_numpy(), cs["coordY"].to_numpy()
    return ((float(np.percentile(x, 10)), float(np.percentile(x, 90))),
            (float(np.percentile(y, 10)), float(np.percentile(y, 90))),
            int(len(cs)))


def _league_platoon(PA, SP_pairs, cutoff):
    """League K%/BB% by batterHand over starter PAs in the calib window
    — the small-sample shrink target. A pre-registered constant."""
    spa = PA.merge(SP_pairs, on="game_id", how="left")
    spa = spa[(spa["pitcherName"] == spa["sp_name"])
              & (spa["date"] < cutoff)]
    lg = {}
    for hand in ("L", "R"):
        sub = spa[spa["batterHand"] == hand]
        lg[hand] = {"K": float((sub["result"] == "SO").mean()),
                    "BB": float(sub["result"].isin(
                        ["uBB", "IBB", "HBP"]).mean())}
    return lg


def _entropy(vc):
    tot = vc.sum()
    if tot <= 0:
        return np.nan
    p = vc[vc > 0] / tot
    return float(-(p * np.log2(p)).sum())


def _compute_pitch_features(G, PA, PI, zx, zy, league):
    """One row per game_id with home_/away_/diff_ pitch-level features.
    EVERY value uses ONLY games strictly before that game."""
    PI = PI.copy()
    PI["is_swing"] = PI["pitchCode"].isin(SWING_CODES)
    PI["is_whiff"] = PI["pitchCode"].isin(WHIFF_CODES)
    PI["in_zone"] = ((PI["coordX"] >= zx[0]) & (PI["coordX"] <= zx[1])
                     & (PI["coordY"] >= zy[0]) & (PI["coordY"] <= zy[1]))
    PI["fps"] = PI["pitchCode"].isin(
        {"S", "SW", "F", "FT", "H", "BUNT"})
    gmeta = G.set_index("game_id")[["date", "g_seq"]]
    sp_long = []
    for _, r in G.iterrows():
        for side in ("home", "away"):
            sp_long.append({"game_id": r["game_id"], "side": side,
                            "sp_name": r[f"{side}_sp_name"]})
    SP = pd.DataFrame(sp_long)
    out = G[["game_id", "date", "g_seq"]].copy()

    # ===== BLOCK 1 — starter stuff / command =========================
    sg = []
    for _, r in SP.iterrows():
        sub = PI[(PI["game_id"] == r["game_id"])
                 & (PI["pitcherName"] == r["sp_name"])]
        if not len(sub):
            continue
        sw = sub["is_swing"]
        sg.append({
            "sp_name": r["sp_name"], "game_id": r["game_id"],
            "side": r["side"], "n_pitch": len(sub),
            "v_sum": sub["velocity"].sum(skipna=True),
            "v_n": int(sub["velocity"].notna().sum()),
            "v_mean_game": sub["velocity"].mean(skipna=True),
            "zone_n": int(sub["in_zone"].sum()),
            "sw_strike_n": int((sw & ~sub["inPlay"]
                                & sub["isStrike"]).sum()),
            "swing_n": int(sw.sum()),
            "fps_n": int(sub["fps"].sum()),
            "mix": sub.loc[sub.pitchType.astype(str).ne(""),
                           "pitchType"].value_counts()})
    SG = pd.DataFrame(sg)
    if len(SG):
        SG = SG.merge(gmeta, on="game_id", how="left").sort_values(
            ["sp_name", "date", "g_seq", "game_id"]).reset_index(
            drop=True)
    feat = []
    for _n, gp in SG.groupby("sp_name"):
        gp = gp.reset_index(drop=True)
        for i in range(len(gp)):
            prev = gp.iloc[max(0, i - ROLL_SP_PL):i]   # strictly prior
            row = {"game_id": gp.iloc[i]["game_id"],
                   "side": gp.iloc[i]["side"]}
            if len(prev) < MIN_PRIOR_PL:
                for c in ("sp_velo", "sp_velo_slope", "sp_mix_entropy",
                          "sp_zone_pct", "sp_swstr_pct", "sp_fps_pct"):
                    row[c] = np.nan
            else:
                vn = prev["v_n"].sum()
                row["sp_velo"] = (prev["v_sum"].sum() / vn
                                  if vn > 0 else np.nan)
                vg = prev["v_mean_game"].to_numpy(dtype=float)
                ok = np.isfinite(vg)
                row["sp_velo_slope"] = (
                    float(np.polyfit(np.arange(len(vg))[ok],
                                     vg[ok], 1)[0])
                    if ok.sum() >= 2 else 0.0)
                mix = prev["mix"].sum()
                row["sp_mix_entropy"] = (_entropy(mix)
                                         if hasattr(mix, "sum")
                                         else np.nan)
                npz = prev["n_pitch"].sum()
                row["sp_zone_pct"] = (prev["zone_n"].sum() / npz
                                      if npz > 0 else np.nan)
                row["sp_swstr_pct"] = (prev["sw_strike_n"].sum() / npz
                                       if npz > 0 else np.nan)
                row["sp_fps_pct"] = (prev["fps_n"].sum() / npz
                                     if npz > 0 else np.nan)
            feat.append(row)
    B1 = pd.DataFrame(feat)

    # ===== BLOCK 2 — starter platoon split (shrunk) ==================
    SPA = PA.merge(SP.rename(columns={"side": "p_side"}),
                   on="game_id", how="left")
    SPA = SPA[SPA["pitcherName"] == SPA["sp_name"]].copy()
    PA2 = PA.copy()
    PA2["bh"] = PA2["batterHand"].where(
        PA2["batterHand"].isin(["L", "R"]))
    lineup_mix = (PA2.dropna(subset=["bh"])
                  .groupby(["game_id", "bat_side"])["bh"]
                  .apply(lambda s: (s == "L").mean())
                  .rename("opp_L_frac").reset_index())
    feat = []
    SPA = SPA.sort_values(["sp_name", "date", "g_seq", "game_id"])
    for _n, gp in SPA.groupby("sp_name"):
        gp = gp.reset_index(drop=True)
        gids = gp["game_id"].drop_duplicates().tolist()
        gpos = {gid: j for j, gid in enumerate(gids)}
        for gid in gids:
            j = gpos[gid]
            prev_gids = gids[max(0, j - ROLL_SP_PL):j]
            side = gp.loc[gp.game_id == gid, "p_side"].iloc[0]
            row = {"game_id": gid, "side": side}
            if len(prev_gids) < MIN_PRIOR_PL:
                row["sp_plat_K"] = row["sp_plat_BB"] = np.nan
            else:
                pv = gp[gp.game_id.isin(prev_gids)]
                lf = lineup_mix[(lineup_mix.game_id == gid)
                                & (lineup_mix.bat_side == side)]
                Lfrac = (float(lf["opp_L_frac"].iloc[0])
                         if len(lf) else 0.5)
                vK = vBB = 0.0
                for hand, w in (("L", Lfrac), ("R", 1 - Lfrac)):
                    hh = pv[pv.batterHand == hand]
                    n = len(hh)
                    kk = (hh["result"] == "SO").sum()
                    bbn = hh["result"].isin(
                        ["uBB", "IBB", "HBP"]).sum()
                    vK += w * ((kk + LEAGUE_SHRINK_K * league[hand]["K"])
                               / (n + LEAGUE_SHRINK_K))
                    vBB += w * ((bbn + LEAGUE_SHRINK_K
                                 * league[hand]["BB"])
                                / (n + LEAGUE_SHRINK_K))
                row["sp_plat_K"] = vK
                row["sp_plat_BB"] = vBB
            feat.append(row)
    B2 = pd.DataFrame(feat)

    # ===== BLOCK 3 — lineup contact quality & discipline =============
    pt = PA.copy()
    pt["hard"] = (pt["hardness"] == "H").astype(float)
    pt["ld"] = (pt["trajectory"] == "L").astype(float)
    pt["gb"] = (pt["trajectory"] == "G").astype(float)
    pt["fb"] = (pt["trajectory"] == "F").astype(float)
    pt["bip"] = pt["hardness"].astype(str).ne("").astype(float)
    pt["K"] = (pt["result"] == "SO").astype(float)
    pt["BB"] = pt["result"].isin(["uBB", "IBB", "HBP"]).astype(float)
    pag = (pt.groupby(["game_id", "bat_team", "bat_side"])
           .agg(hard=("hard", "sum"), ld=("ld", "sum"),
                gb=("gb", "sum"), fb=("fb", "sum"),
                bip=("bip", "sum"), K=("K", "sum"),
                BB=("BB", "sum"), PA=("result", "size"))
           .reset_index())
    pb = PI.copy()
    pb["chase"] = (pb["is_swing"] & ~pb["in_zone"]).astype(float)
    pb["oz"] = (~pb["in_zone"]).astype(float)
    pb["zcon"] = (pb["in_zone"] & pb["inPlay"]).astype(float)
    pb["zn"] = pb["in_zone"].astype(float)
    pitg = (pb.groupby(["game_id", "bat_team", "bat_side"])
            .agg(chase=("chase", "sum"), oz=("oz", "sum"),
                 zcon=("zcon", "sum"), zn=("zn", "sum"))
            .reset_index())
    TG = (pag.merge(pitg, on=["game_id", "bat_team", "bat_side"],
                    how="left")
          .merge(gmeta, on="game_id", how="left")
          .sort_values(["bat_team", "date", "g_seq", "game_id"]))
    feat = []
    for _t, gp in TG.groupby("bat_team"):
        gp = gp.reset_index(drop=True)
        for i in range(len(gp)):
            prev = gp.iloc[max(0, i - ROLL_BAT_PL):i]
            row = {"game_id": gp.iloc[i]["game_id"],
                   "side": gp.iloc[i]["bat_side"]}
            if len(prev) < MIN_PRIOR_PL:
                for c in ("bat_hardhit", "bat_ld", "bat_gbfb",
                          "bat_chase", "bat_zcontact", "bat_K",
                          "bat_BB"):
                    row[c] = np.nan
            else:
                bip = prev["bip"].sum()
                pa_ = prev["PA"].sum()
                oz = prev["oz"].sum()
                zn = prev["zn"].sum()
                fb = prev["fb"].sum()
                row["bat_hardhit"] = (prev["hard"].sum() / bip
                                      if bip > 0 else np.nan)
                row["bat_ld"] = (prev["ld"].sum() / bip
                                 if bip > 0 else np.nan)
                row["bat_gbfb"] = (prev["gb"].sum() / fb
                                   if fb > 0 else np.nan)
                row["bat_chase"] = (prev["chase"].sum() / oz
                                    if oz > 0 else np.nan)
                row["bat_zcontact"] = (prev["zcon"].sum() / zn
                                       if zn > 0 else np.nan)
                row["bat_K"] = (prev["K"].sum() / pa_
                                if pa_ > 0 else np.nan)
                row["bat_BB"] = (prev["BB"].sum() / pa_
                                 if pa_ > 0 else np.nan)
            feat.append(row)
    B3 = pd.DataFrame(feat)

    # ===== BLOCK 4 — bullpen pitch quality (relief only) =============
    sp_by_game = {}
    for _, r in SP.iterrows():
        sp_by_game.setdefault(r["game_id"], {})[r["side"]] = r["sp_name"]
    PIp = PI.copy()
    PIp["pit_side"] = np.where(PIp["bat_side"] == "home",
                               "away", "home")

    def _is_relief(rr):
        st = sp_by_game.get(rr["game_id"], {}).get(rr["pit_side"])
        return rr["pitcherName"] != st
    PIp["relief"] = PIp.apply(_is_relief, axis=1)
    pen = PIp[PIp["relief"]].copy()
    pen["on_base_proxy"] = (pen["inPlay"]
                            | ~pen["isStrike"]).astype(float)
    peng = (pen.groupby(["game_id", "pit_side"])
            .agg(v_sum=("velocity", "sum"),
                 v_n=("velocity", lambda s: int(s.notna().sum())),
                 sw_strike=("pitchCode",
                            lambda s: int(s.isin(WHIFF_CODES).sum())),
                 npz=("pitchCode", "size"))
            .reset_index())
    peng = peng.merge(
        pen.groupby(["game_id", "pit_side"]).agg(
            base=("on_base_proxy", "mean")).reset_index(),
        on=["game_id", "pit_side"], how="left")
    gt = []
    for _, r in G.iterrows():
        gt.append({"game_id": r["game_id"], "pit_side": "home",
                   "pen_team": r["home_team"]})
        gt.append({"game_id": r["game_id"], "pit_side": "away",
                   "pen_team": r["away_team"]})
    peng = (peng.merge(pd.DataFrame(gt),
                       on=["game_id", "pit_side"], how="left")
            .merge(gmeta, on="game_id", how="left")
            .sort_values(["pen_team", "date", "g_seq", "game_id"]))
    feat = []
    for _t, gp in peng.groupby("pen_team"):
        gp = gp.reset_index(drop=True)
        for i in range(len(gp)):
            prev = gp.iloc[max(0, i - ROLL_PEN_PL):i]
            row = {"game_id": gp.iloc[i]["game_id"],
                   "side": gp.iloc[i]["pit_side"]}
            if len(prev) < MIN_PRIOR_PL:
                for c in ("pen_velo", "pen_swstr", "pen_onbase"):
                    row[c] = np.nan
            else:
                vn = prev["v_n"].sum()
                npz = prev["npz"].sum()
                row["pen_velo"] = (prev["v_sum"].sum() / vn
                                   if vn > 0 else np.nan)
                row["pen_swstr"] = (prev["sw_strike"].sum() / npz
                                    if npz > 0 else np.nan)
                row["pen_onbase"] = float(prev["base"].mean())
            feat.append(row)
    B4 = pd.DataFrame(feat)

    # ===== BLOCK 5 — form-ized WPA/RE24 (strictly prior) =============
    # NAMING IS A LEAKAGE GUARD: *_prior_s2d only; never bare WPA/RE24.
    bg = (PA.groupby(["batterName", "game_id", "date", "g_seq",
                      "bat_team", "bat_side"])
          .agg(WPA=("WPA", "sum"), RE24=("RE24", "sum"),
               nPA=("WPA", "size")).reset_index())
    bg = bg.sort_values(["batterName", "date", "g_seq", "game_id"])
    for src_c, dst in (("WPA", "cum_WPA"), ("RE24", "cum_RE24"),
                       ("nPA", "cum_PA")):
        bg[dst] = (bg.groupby("batterName")[src_c]
                   .apply(lambda s: s.shift(1).cumsum())
                   .reset_index(level=0, drop=True))
    bg["batter_WPA_per_PA_prior_s2d"] = bg["cum_WPA"] / bg["cum_PA"]
    bg["batter_RE24_per_PA_prior_s2d"] = bg["cum_RE24"] / bg["cum_PA"]
    tg = (bg.groupby(["game_id", "bat_team", "bat_side"])
          .agg(team_WPA_prior_s2d=("batter_WPA_per_PA_prior_s2d",
                                   "mean"),
               team_RE24_prior_s2d=("batter_RE24_per_PA_prior_s2d",
                                    "mean"))
          .reset_index())
    B5 = tg.rename(columns={"bat_side": "side"})[
        ["game_id", "side", "team_WPA_prior_s2d",
         "team_RE24_prior_s2d"]]

    def _widen(B, cols):
        nonlocal out
        added = []
        for c in cols:
            hh = B[B.side == "home"][["game_id", c]].rename(
                columns={c: f"home_{c}"})
            aa = B[B.side == "away"][["game_id", c]].rename(
                columns={c: f"away_{c}"})
            out = out.merge(hh, on="game_id", how="left").merge(
                aa, on="game_id", how="left")
            out[f"diff_{c}"] = out[f"home_{c}"] - out[f"away_{c}"]
            added.append(f"diff_{c}")
        return added

    pl = []
    pl += _widen(B1, ["sp_velo", "sp_velo_slope", "sp_mix_entropy",
                      "sp_zone_pct", "sp_swstr_pct", "sp_fps_pct"])
    pl += _widen(B2, ["sp_plat_K", "sp_plat_BB"])
    pl += _widen(B3, ["bat_hardhit", "bat_ld", "bat_gbfb",
                      "bat_chase", "bat_zcontact", "bat_K", "bat_BB"])
    pl += _widen(B4, ["pen_velo", "pen_swstr", "pen_onbase"])
    pl += _widen(B5, ["team_WPA_prior_s2d", "team_RE24_prior_s2d"])
    return out, pl


# ---- run -------------------------------------------------------------
_src = _discover_sources(ROOT / "data/raw")
G, PA, PI = _build_long(_src)
SP_pairs = pd.concat([
    G[["game_id"]].assign(sp_name=G["home_sp_name"]),
    G[["game_id"]].assign(sp_name=G["away_sp_name"])],
    ignore_index=True)
CUT = _calib_cutoff(G)
ZX, ZY, N_CS = _strike_zone(PI, CUT)
LEAGUE = _league_platoon(PA, SP_pairs, CUT)
print(f"STEP 3e: games={len(G)} PAs={len(PA)} pitch-events={len(PI)}")
print(f"FROZEN calibration cutoff = {CUT.date()}  "
      f"(constants derived ONLY from games strictly before it)")
print(f"FROZEN empirical strike zone (pre-registered modelling "
      f"constant):")
print(f"  coordX in [{ZX[0]:.2f}, {ZX[1]:.2f}]  "
      f"coordY in [{ZY[0]:.2f}, {ZY[1]:.2f}]  "
      f"(from {N_CS} called strikes, pitchCode=='S')")
print(f"FROZEN platoon league shrink target: {LEAGUE}")
pf, PITCH_FEATS = _compute_pitch_features(G, PA, PI, ZX, ZY, LEAGUE)

# Merge into the SAME model-ready frame STEP 3d/4 consume. Keyed on the
# IDENTICAL game_id formula STEP-1 used -> additive integration.
mr = pd.read_csv(ROOT / "data/processed/model_ready_v2.csv",
                 parse_dates=["date"])
_n0 = len(mr)
keep = ["game_id"] + [c for c in pf.columns
                      if c.startswith(("home_", "away_", "diff_"))]
mr = mr.merge(pf[keep], on="game_id", how="left")
assert len(mr) == _n0, ("STEP 3e merge changed row count "
                         f"({_n0} -> {len(mr)}) — game_id mismatch")
matched = mr[PITCH_FEATS].notna().any(axis=1).sum()
print(f"\nmerged {len(PITCH_FEATS)} pitch-level diff features into "
      f"model_ready_v2.csv ({matched}/{len(mr)} rows got >=1 "
      f"pitch feature; rest cold-start -> STEP 4 median-imputes)")

# features_complete is DELIBERATELY NOT updated: pitch-level NaN flows
# through STEP 4's SimpleImputer(median). The ablation's two arms MUST
# run on the same N (same post-saber-warmup rows) to be apples-to-apples.
assert "features_complete" in mr.columns
print("features_complete left UNCHANGED (ablation arms share N): "
      f"sum={int(mr['features_complete'].sum())}/{len(mr)}")

# ---- range / sanity asserts (executable, in-notebook) ---------------
for side in ("home", "away"):
    for stub in ("sp_zone_pct", "sp_swstr_pct", "sp_fps_pct",
                 "bat_hardhit", "bat_ld", "bat_chase",
                 "bat_zcontact", "bat_K", "bat_BB", "sp_plat_K",
                 "sp_plat_BB", "pen_swstr", "pen_onbase"):
        col = f"{side}_{stub}"
        v = mr[col].dropna()
        if len(v):
            assert v.min() >= -1e-9 and v.max() <= 1 + 1e-9, (
                f"rate feature {col} out of [0,1]: "
                f"[{v.min():.4f},{v.max():.4f}]")
for side in ("home", "away"):
    e = mr[f"{side}_sp_mix_entropy"].dropna()
    if len(e):
        assert e.min() >= -1e-9, (
            f"{side}_sp_mix_entropy negative: {e.min()}")
print("range asserts OK: all rate features in [0,1], "
      "pitch-mix entropy >= 0")

# HARD LEAKAGE-NAMING GUARD (executable): every WPA/RE24-derived column
# MUST end _prior_s2d; bare WPA/RE24/homeWE/RE never enter as features.
_pl_cols = [c for c in mr.columns
            if c.startswith(("home_", "away_", "diff_"))
            and any(k in c for k in ("WPA", "RE24", "homeWE"))]
for c in _pl_cols:
    assert c.endswith("_prior_s2d"), (
        f"LEAKAGE-NAMING VIOLATION: {c!r} carries WPA/RE24/homeWE but "
        "does not end _prior_s2d — within-game state must only appear "
        "as a strictly-prior per-player rolling average")
print(f"naming guard OK: WPA/RE24 cols all *_prior_s2d ({_pl_cols})")

mr.to_csv(ROOT / "data/processed/model_ready_v2.csv", index=False,
          encoding="utf-8")
print("rewritten model_ready_v2.csv shape", mr.shape)

# Append the EXPLICIT pitch list to _feature_lists.json (STEP 4 reads
# saber + pitch_level explicitly — no prefix detection in the ablation).
_flp = EVALV2 / "_feature_lists.json"
_fl = (json.loads(_flp.read_text(encoding="utf-8"))
       if _flp.exists() else {"saber": [], "pitch_level": []})
_fl["pitch_level"] = PITCH_FEATS
_fl["strike_zone"] = {"coordX": list(ZX), "coordY": list(ZY),
                      "n_called_strikes": N_CS,
                      "rule": "p10..p90 of coordX&coordY over "
                              "pitchCode=='S' (called strikes) in the "
                              "frozen calibration window",
                      "calib_cutoff": str(CUT.date())}
_fl["platoon_league_shrink_target"] = LEAGUE
_flp.write_text(json.dumps(_fl, indent=2, ensure_ascii=False),
                encoding="utf-8")
print("persisted pitch feature list + frozen constants:", _flp)

## STEP 3d — 標準化 ＋ 時間感知切分（**絕不隨機**）

沿用 v1 的切分哲學：訓練＝最早的時間區塊、valid＝中段、test＝最後的比賽
（季末＋季後賽）。標準化在建模 pipeline 內以 `StandardScaler` 完成（只 fit
train，避免洩漏）。STEP 4 的 walk-forward `TimeSeriesSplit` 也是時間感知的。

In [ ]:
# === STEP 3d — standardise note + time-aware split =================
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data/processed/model_ready_v2.csv", parse_dates=["date"])
TARGET = "is_home_win"
df = df[df["features_complete"] == 1].reset_index(drop=True)
print(f"post warm-up: {len(df)} games  "
      f"home-win base={df[TARGET].mean():.3f}")

# Time-aware split (NEVER random) — mirrors v1's split philosophy.
# 2023 whole season + early 2024 = train; mid-2024 = valid;
# season-end + post-season 2024 = test (latest games).
train = df[df["date"] < "2024-08-01"].reset_index(drop=True)
valid = df[(df["date"] >= "2024-08-01")
           & (df["date"] < "2024-09-16")].reset_index(drop=True)
test = df[df["date"] >= "2024-09-16"].reset_index(drop=True)


def _rng(d):
    return (f"{d['date'].min().date()}..{d['date'].max().date()}"
            if len(d) else "EMPTY")


print(f"split sizes  train={len(train)}  valid={len(valid)}  "
      f"test={len(test)}")
print(f"train  {_rng(train)}  home-win={train[TARGET].mean():.3f}")
print(f"valid  {_rng(valid)}  home-win={valid[TARGET].mean():.3f}")
print(f"test   {_rng(test)}  home-win={test[TARGET].mean():.3f}")
print("standardisation is done INSIDE the modelling pipeline "
      "(StandardScaler fit on training folds only — no split leakage); "
      "STEP 4's TimeSeriesSplit walk-forward is likewise time-aware.")

## STEP 4 — overturn ablation（walk-forward season-OOF ＋ stacking ＋ 護欄）

跑論文的模型 menu：Logistic Regression、Decision Tree、Random Forest、
XGBoost、LightGBM——用 **walk-forward season-OOF**（`TimeSeriesSplit`，
沿用 v1 的 `ts_oof_proba` pattern）評估，**不**用論文的隨機 5-fold。

> 為什麼不用論文的隨機 5-fold：運動資料是時間序列，隨機 fold 會把「未來」
> 比賽放進訓練、用來預測「過去」，rolling 特徵又跨越 fold 邊界——隨機 CV
> 因此**系統性高估**，這正是 0.97 之外的第二層膨脹來源。時間感知的
> walk-forward（只用過去訓練、預測未來）才誠實。

### 推翻檢定（OVERTURN ABLATION，本步的科學核心）

對**同樣的模型機具、同樣的 walk-forward season-OOF、同樣的 N**，跑**兩個
特徵集**並比較：

- **(i) sabermetric-only** ＝ 既有 v1 leak-free 特徵集（鎖定的 ≈0.53 基準）。
- **(ii) + pitch-level** ＝ (i) ∪ STEP 3e 的逐球賽前區塊。

兩臂的特徵清單**由 `Results/v2/eval/_feature_lists.json` 明確讀取**
（`saber` / `pitch_level`），**不**靠前綴猜測，避免錯欄混進錯臂。再對
**兩個集合各跑一個 stacking 區塊**（base = LightGBM ＋ RandomForest →
meta = LogisticRegression，皆 walk-forward）。

每個模型的 (i) vs (ii) season-OOF AUC 以 **paired bootstrap（~1000 次，
模型內兩臂用相同的 OOF row index）** 取 95% CI，畫成
`Results/v2/figures/pitch_level_ablation.png` 分組長條（含 CI）。

**誠實判讀（不可調參、不可為衝數字放寬任何 leak-free 規則）**：leak-free
天花板現實上 ≈0.57–0.60（Vegas／學術）。若 (ii) 超過 (i) **且 CI 不重疊**
→ 對「除主場優勢外無訊號」是**真正的推翻**，幅度雖小但真實且可發表；
若不然，負面結論延伸到 pitch-level。**不**朝任何目標數字調參。

**硬護欄（可執行程式碼，不是註解）**：每個模型（**兩臂都算**）算完
season-OOF AUC 後立即 `assert auc <= 0.70, ...`；兩個 stacking 也各有
`assert stack_auc <= 0.70`。若任何 leak-free 模型（含 pitch-level
augmented 與 stacking）衝破 0.70，代表特徵被 same-game 污染，notebook
**當場 raise** 停住。論文的 0.97 會觸發它；v1 最差折 0.584 不會。

In [ ]:
# === STEP 4 — paper model menu, walk-forward season-OOF + stacking =
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore")
RNG = 42
ROOT = Path.cwd()
EVALV2 = ROOT / "Results/v2/eval"
EVALV2.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "data/processed/model_ready_v2.csv", parse_dates=["date"])
TARGET = "is_home_win"
# features_complete is the SABER cold-start mask. Both ablation arms run
# on EXACTLY these rows (same N) — pitch-level NaN flows through the
# median imputer. This keeps (i) vs (ii) apples-to-apples.
df = df[df["features_complete"] == 1].reset_index(drop=True)

# EXPLICIT feature lists (STEP 3b wrote saber, STEP 3e appended pitch).
# Never prefix-detect for the ablation arms — that risks the wrong
# column landing in the wrong arm. Fall back to prefix only if the file
# is missing (STEP 3e not run in this kernel) and warn loudly.
_flp = EVALV2 / "_feature_lists.json"
if _flp.exists():
    _FL = json.loads(_flp.read_text(encoding="utf-8"))
    SABER = [c for c in _FL.get("saber", []) if c in df.columns]
    PITCH = [c for c in _FL.get("pitch_level", []) if c in df.columns]
else:
    print("WARNING: _feature_lists.json missing — STEP 3e likely not "
          "run; falling back to prefix detection (saber only, NO "
          "pitch-level ablation possible)")
    SABER = [c for c in df.columns if c.startswith("diff_")
             and c not in ("diff_score",)
             and not any(k in c for k in (
                 "sp_velo", "sp_mix", "sp_zone", "sp_swstr", "sp_fps",
                 "sp_plat", "bat_hardhit", "bat_ld", "bat_gbfb",
                 "bat_chase", "bat_zcontact", "bat_K", "bat_BB",
                 "pen_velo", "pen_swstr", "pen_onbase", "prior_s2d"))]
    PITCH = []
FEAT_SETS = {"saber": SABER}
if PITCH:
    FEAT_SETS["saber+pitch"] = SABER + PITCH
print(f"post warm-up N={len(df)}")
print(f"(i)  sabermetric-only   : {len(SABER)} feats")
print(f"(ii) + pitch-level      : "
      f"{len(SABER) + len(PITCH)} feats ({len(PITCH)} pitch-level)"
      if PITCH else "(ii) UNAVAILABLE — pitch list empty")
y = df[TARGET]

# Read the STEP-3a leakage-demo numbers back from disk (Colab-robust:
# does not depend on STEP-3a's in-memory LEAK_DEMO still being live, so
# re-running STEP 4 alone or a partial STEP-3 re-run never loses them).
_ld = EVALV2 / "_leak_demo.json"
if _ld.exists():
    LEAK_DEMO = json.loads(_ld.read_text(encoding="utf-8"))
elif "LEAK_DEMO" not in dir():
    LEAK_DEMO = {"note": "STEP 3a not run in this kernel — "
                 "_leak_demo.json missing"}


def make_pipe(estimator, scale):
    """median-impute always (cold-start NaN = league fallback);
    StandardScaler only for the linear model."""
    steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("sc", StandardScaler()))
    steps.append(("clf", estimator))
    return Pipeline(steps)


def ts_oof_proba(estimator, X, y, splitter):
    """Walk-forward out-of-fold P(y=1). TimeSeriesSplit is NOT a
    partition (the first training block is never a test fold), so
    cross_val_predict rejects it. Do it by hand: clone + fit on each
    fold's past, predict its future. Rows never in any test fold stay
    NaN; the caller masks them. (Reused verbatim from v1's step3.)"""
    X = X.reset_index(drop=True)
    y = pd.Series(np.asarray(y))
    oof = np.full(len(y), np.nan)
    for tr, te in splitter.split(X):
        est = clone(estimator)
        est.fit(X.iloc[tr], y.iloc[tr])
        oof[te] = est.predict_proba(X.iloc[te])[:, 1]
    return oof


tscv = TimeSeriesSplit(n_splits=5)

# ---- the paper's 5-model menu (LR, DT, RF, XGB, LGBM) ----------------
models = {
    "LogisticRegression": (make_pipe(
        LogisticRegression(max_iter=2000, C=1.0, solver="liblinear"),
        scale=True)),
    "DecisionTree": (make_pipe(
        DecisionTreeClassifier(max_depth=4, min_samples_leaf=20,
                               random_state=RNG), scale=False)),
    "RandomForest": (make_pipe(
        RandomForestClassifier(n_estimators=400, max_depth=6,
                               min_samples_leaf=5, random_state=RNG,
                               n_jobs=-1), scale=False)),
    "XGBoost": (make_pipe(
        xgb.XGBClassifier(n_estimators=400, max_depth=3,
                          learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, reg_alpha=0.1,
                          reg_lambda=1.0, eval_metric="logloss",
                          random_state=RNG, n_jobs=-1,
                          tree_method="hist"), scale=False)),
    "LightGBM": (make_pipe(
        lgb.LGBMClassifier(n_estimators=400, num_leaves=15,
                           learning_rate=0.05, subsample=0.8,
                           colsample_bytree=0.8, min_child_samples=10,
                           reg_alpha=0.1, reg_lambda=1.0,
                           random_state=RNG, n_jobs=-1,
                           verbosity=-1), scale=False)),
}

def _boot_ci(yv, pv, n_boot=1000, seed=RNG):
    """Percentile bootstrap 95% CI of AUC over OOF rows."""
    rng = np.random.default_rng(seed)
    yv = np.asarray(yv)
    pv = np.asarray(pv)
    n = len(yv)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(yv[idx])) < 2:
            continue
        aucs.append(roc_auc_score(yv[idx], pv[idx]))
    if not aucs:
        return (float("nan"), float("nan"))
    return (float(np.percentile(aucs, 2.5)),
            float(np.percentile(aucs, 97.5)))


# ===================================================================
# OVERTURN ABLATION — same machinery, same N, two feature sets:
#   (i)  saber  = locked v1 leak-free baseline (~0.53)
#   (ii) saber+pitch = (i) U STEP-3e pitch-level blocks
# Per model: paired bootstrap CI uses the SAME OOF row indices for
# both arms (rows scored in BOTH arms) -> the comparison is paired.
# ===================================================================
ablation = {}                       # model -> {set -> metrics}
results = []                        # flat rows for results_v2.csv
stack_results = {}
for set_name, feats in FEAT_SETS.items():
    Xs = df[feats]
    print("\n" + "=" * 68)
    print(f"FEATURE SET ({set_name}) — {len(feats)} features — "
          f"walk-forward season-OOF (NOT random 5-fold)")
    print("=" * 68)
    base_oof = {}
    set_oofs = {}
    for name, pipe in models.items():
        oof = ts_oof_proba(pipe, Xs, y, tscv)
        k = ~np.isnan(oof)
        auc = (roc_auc_score(y[k], oof[k])
               if y[k].nunique() > 1 else float("nan"))
        # ---- HARD LEAKAGE GUARDRAIL (fires in-loop, BOTH arms) -----
        assert auc <= 0.70, f"LEAKAGE SUSPECTED ({name} [{set_name}] season-OOF AUC={auc:.3f} > 0.70) — STOP and audit features for same-game contamination"
        folds = []
        for _tr, _te in tscv.split(Xs):
            yt = y.values[_te]
            folds.append(round(float(roc_auc_score(yt, oof[_te])), 3)
                         if len(np.unique(yt)) > 1 else float("nan"))
        ablation.setdefault(name, {})[set_name] = {
            "season_oof_auc": round(float(auc), 4),
            "n_scored": int(k.sum()), "folds": folds}
        results.append({"feature_set": set_name, "model": name,
                        "season_oof_auc": round(float(auc), 4),
                        "n_scored": int(k.sum())})
        set_oofs[name] = oof
        if name in ("LightGBM", "RandomForest"):
            base_oof[name] = oof
        print(f"  {name:20s} season-OOF AUC={auc:.4f}  "
              f"n={int(k.sum())}  folds={folds}")

    # ---- stacking on THIS feature set --------------------------------
    print(f"  -- STACKING [{set_name}] base [LGBM,RF] -> meta LR --")
    M = np.column_stack([base_oof["LightGBM"], base_oof["RandomForest"]])
    meta_mask = ~np.isnan(M).any(axis=1)
    Mk = M[meta_mask]
    yk = y.values[meta_mask]
    meta_oof = ts_oof_proba(LogisticRegression(max_iter=2000),
                            pd.DataFrame(Mk), yk, tscv)
    mk = ~np.isnan(meta_oof)
    stack_auc = (roc_auc_score(yk[mk], meta_oof[mk])
                 if len(np.unique(yk[mk])) > 1 else float("nan"))
    assert stack_auc <= 0.70, f"LEAKAGE SUSPECTED (Stacking [{set_name}] season-OOF AUC={stack_auc:.3f} > 0.70) — STOP and audit features for same-game contamination"
    sname = "Stacking(LGBM+RF->LR)"
    ablation.setdefault(sname, {})[set_name] = {
        "season_oof_auc": round(float(stack_auc), 4),
        "n_scored": int(mk.sum()), "folds": None}
    results.append({"feature_set": set_name, "model": sname,
                    "season_oof_auc": round(float(stack_auc), 4),
                    "n_scored": int(mk.sum())})
    # full-row meta OOF aligned to df index for paired CI
    _full = np.full(len(y), np.nan)
    _idx = np.where(meta_mask)[0]
    _full[_idx[mk]] = meta_oof[mk]
    set_oofs[sname] = _full
    stack_results[set_name] = stack_auc
    print(f"  {sname} season-OOF AUC={stack_auc:.4f}  n={int(mk.sum())}")
    print("  NOTE: stacking reduces variance; it cannot manufacture "
          "signal the base learners lack.")
    globals()[f"_oofs_{set_name.replace('+', '_')}"] = set_oofs

# ---- paired bootstrap CI + overturn read -----------------------------
ABL_MODELS = list(models.keys()) + ["Stacking(LGBM+RF->LR)"]
HAS_II = "saber+pitch" in FEAT_SETS
ci_table = {}
overturn_rows = []
for name in ABL_MODELS:
    oi = globals()["_oofs_saber"][name]
    rec = {"saber": ablation[name]["saber"]["season_oof_auc"]}
    if HAS_II:
        oii = globals()["_oofs_saber_pitch"][name]
        paired = ~np.isnan(oi) & ~np.isnan(oii)   # SAME rows both arms
        yv = y.values[paired]
        lo_i, hi_i = _boot_ci(yv, oi[paired])
        lo_ii, hi_ii = _boot_ci(yv, oii[paired])
        a_i = roc_auc_score(yv, oi[paired])
        a_ii = roc_auc_score(yv, oii[paired])
        overturn = bool(a_ii > a_i and lo_ii > hi_i)   # CI disjoint
        rec.update({"saber+pitch":
                    ablation[name]["saber+pitch"]["season_oof_auc"],
                    "paired_n": int(paired.sum()),
                    "auc_saber_paired": round(float(a_i), 4),
                    "auc_pitch_paired": round(float(a_ii), 4),
                    "ci_saber": [round(lo_i, 4), round(hi_i, 4)],
                    "ci_pitch": [round(lo_ii, 4), round(hi_ii, 4)],
                    "overturn_disjoint_ci": overturn})
        overturn_rows.append((name, overturn))
    ci_table[name] = rec

res_df = pd.DataFrame(results)
print("\nABLATION RESULTS (walk-forward season-OOF AUC):")
print(res_df.to_string(index=False))
res_df.to_csv(EVALV2 / "results_v2.csv", index=False)

# ---- grouped-bar figure with bootstrap 95% CI ------------------------
if HAS_II:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    FIGV2 = ROOT / "Results/v2/figures"
    FIGV2.mkdir(parents=True, exist_ok=True)
    labels = ABL_MODELS
    xi = np.arange(len(labels))
    w = 0.38
    ai = [ci_table[m]["auc_saber_paired"] for m in labels]
    aii = [ci_table[m]["auc_pitch_paired"] for m in labels]
    ei = [[ci_table[m]["auc_saber_paired"] - ci_table[m]["ci_saber"][0]
           for m in labels],
          [ci_table[m]["ci_saber"][1] - ci_table[m]["auc_saber_paired"]
           for m in labels]]
    eii = [[ci_table[m]["auc_pitch_paired"] - ci_table[m]["ci_pitch"][0]
            for m in labels],
           [ci_table[m]["ci_pitch"][1] - ci_table[m]["auc_pitch_paired"]
            for m in labels]]
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar(xi - w / 2, ai, w, yerr=ei, capsize=4, color="#2471a3",
           label="(i) sabermetric-only")
    ax.bar(xi + w / 2, aii, w, yerr=eii, capsize=4, color="#c0392b",
           label="(ii) + pitch-level")
    ax.axhline(0.5, ls="--", c="k", alpha=.5, label="chance 0.50")
    ax.axhspan(0.57, 0.60, color="#27ae60", alpha=.12,
               label="leak-free ceiling ~0.57-0.60 (Vegas/academic)")
    ax.set_xticks(xi)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylabel("walk-forward season-OOF AUC (paired rows)")
    ax.set_ylim(0.40, 0.72)
    ax.set_title("Pitch-level overturn ablation — season-OOF AUC with "
                 "paired bootstrap 95% CI\n(same machinery, same N; "
                 "(ii) beats (i) beyond disjoint CI => genuine overturn)")
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGV2 / "pitch_level_ablation.png", dpi=130)
    plt.close(fig)
    print("saved", FIGV2 / "pitch_level_ablation.png")
    any_ot = any(o for _n, o in overturn_rows)
    print("\nHONEST READING: leak-free realistic ceiling ~0.57-0.60. "
          + ("AT LEAST ONE model shows (ii)>(i) with DISJOINT 95% CI "
             "-> a genuine, modest-but-real overturn of 'no signal "
             "beyond home-field advantage'; publishable as a positive "
             "leak-free result."
             if any_ot else
             "NO model shows (ii)>(i) beyond overlapping CI -> the "
             "negative result EXTENDS to pitch-level: even pitch-by-"
             "pitch pre-game information does not beat the home-field "
             "intercept. Consistent with v1's locked finding."))

_saber_oof = {m: ablation[m]["saber"]["season_oof_auc"]
              for m in ablation}
_saber_folds = {m: ablation[m]["saber"]["folds"] for m in ablation}
_metrics = {
    "design": "leak-free, prediction-oriented copy of Lo et al. 2025",
    "evaluation": "walk-forward season-OOF (TimeSeriesSplit n=5); "
                  "NOT the paper's random 5-fold",
    "leakage_demo": LEAK_DEMO,
    "n_post_warmup": int(len(df)),
    "home_win_base_rate": float(y.mean()),
    "leakfree_features": SABER,
    # back-compat keys: 'model_season_oof' = the saber-only arm (the
    # locked ~0.53 baseline), same meaning as before the ablation.
    "model_season_oof": _saber_oof,
    "model_oof_folds": _saber_folds,
    "leakage_guardrail": "assert season-OOF AUC <= 0.70 per model in "
                         "BOTH arms + both stacks (fires in-loop; "
                         "paper's 0.97 would trip it, v1's worst fold "
                         "0.584 does not)",
    # ---- NEW: the overturn ablation ----------------------------------
    "pitch_level_ablation": {
        "available": bool(HAS_II),
        "feature_sets": {
            "saber": {"n_features": len(SABER), "features": SABER},
            "saber+pitch": {
                "n_features": len(SABER) + len(PITCH),
                "pitch_level_features": PITCH}},
        "season_oof_auc": {
            m: {s: ablation[m][s]["season_oof_auc"]
                for s in ablation[m]} for m in ablation},
        "paired_bootstrap_ci": ci_table,
        "bootstrap": "1000 resamples, percentile 2.5/97.5; paired "
                     "(same OOF row indices for both arms within a "
                     "model)",
        "overturn_rule": "(ii) auc > (i) auc AND 95% CIs disjoint "
                         "(lo_ii > hi_i)",
        "overturn_any_model": (bool(any(o for _n, o in overturn_rows))
                               if HAS_II else None),
        "figure": "Results/v2/figures/pitch_level_ablation.png"},
    "interpretation": (
        "Sabermetric-only collapses to ~0.50-0.53 season-OOF "
        "(v1's locked negative result). The pitch-level ablation "
        "tests whether strictly pre-game pitch-by-pitch features "
        "overturn that: " + (
            "at least one model beats the saber arm beyond disjoint "
            "95% CI -> a modest but genuine, publishable overturn."
            if (HAS_II and any(o for _n, o in overturn_rows))
            else "no model beats the saber arm beyond overlapping CI "
                 "-> the negative result extends to pitch-level too.")
        + " The 0.97-vs-0.53 leakage_demo gap remains the headline "
          "methodological contribution."),
}
(EVALV2 / "_final_metrics_v2.json").write_text(
    json.dumps(_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8")
print("\nwritten", EVALV2 / "_final_metrics_v2.json")

In [ ]:
# === Cell 11 — RESULTS: v2 metrics JSON + v2 figures ==============
import json
import pathlib
from IPython.display import Image, display

p = pathlib.Path("Results/v2/eval/_final_metrics_v2.json")
if p.exists():
    d = json.loads(p.read_text(encoding="utf-8"))
    print(json.dumps(d, indent=2, ensure_ascii=False))
    print("\n>>> LEAKAGE DEMO  same-game wOBA AUC =",
          d["leakage_demo"]["samegame_wOBA_auc"],
          " vs  pre-game rolling wOBA AUC =",
          d["leakage_demo"]["pregame_roll_wOBA_auc"])
    print(">>> saber-only model season-OOF:", d["model_season_oof"])
    _abl = d.get("pitch_level_ablation", {})
    if _abl.get("available"):
        print(">>> OVERTURN ABLATION season-OOF (saber vs +pitch):")
        for m, ss in _abl["season_oof_auc"].items():
            print(f"      {m:24s} {ss}")
        print(">>> overturn (any model, disjoint 95% CI)?",
              _abl.get("overturn_any_model"))
    else:
        print(">>> pitch-level ablation UNAVAILABLE "
              "(STEP 3e not run this kernel)")
else:
    print("X _final_metrics_v2.json not generated — see STEP 4 above")

for f in ["leakage_demo.png", "eda_distributions.png", "eda_corr.png",
          "eda_pca.png", "pitch_level_ablation.png"]:
    fp = pathlib.Path(f"Results/v2/figures/{f}")
    if fp.exists():
        print("\n" + str(fp))
        display(Image(str(fp)))
    else:
        print("MISSING", fp)

### Cell 12（選用）打包 / 推回 **只** `Results/v2`

只想下載 → 跑前半（zip）。要推回 → 填 PAT 跑後半。**輸出隔離**：本 cell
只 `git add -f Results/v2`，**絕不**碰 `Results/eval`、`Results/figures`、
v1 notebook 或鎖定報告。

In [ ]:
# === Cell 12（選用）打包 + 可選推回（只 Results/v2）==============
import shutil, subprocess
shutil.make_archive("cpbl_v2_artifacts", "zip", ".", base_dir="Results/v2")
print("-> cpbl_v2_artifacts.zip （只含 Results/v2）可從左側面板下載")

PUSH = False                    # 改 True 並填 PAT 才會推回
if PUSH:
    from getpass import getpass
    tok = getpass("GitHub PAT (repo scope): ")
    REPO = "https://github.com/jiangjiangian/data_science_final_project.git"
    # The REMOTE branch (NOT the local Mac worktree suffix '...-fix2',
    # which never exists on origin — cloning it fails with exit 128).
    BR = "claude/setup-main-agent-BhYTE"
    subprocess.run(["rm", "-rf", "_pushrepo"], check=False)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BR,
                    REPO, "_pushrepo"], check=True)
    subprocess.run(["mkdir", "-p", "_pushrepo/Results/v2"], check=True)
    subprocess.run(["cp", "-r", "Results/v2/eval", "Results/v2/figures",
                    "_pushrepo/Results/v2/"], check=True)
    # OUTPUT ISOLATION: stage ONLY Results/v2 — never Results/eval,
    # Results/figures, the v1 notebook, or the locked report.
    subprocess.run(["git", "-C", "_pushrepo", "add", "-f",
                    "Results/v2"], check=True)
    subprocess.run(["git", "-C", "_pushrepo", "-c", "user.email=colab@run",
                    "-c", "user.name=colab", "commit", "-m",
                    "data: v2 leak-free pipeline artifacts (Results/v2 only)"],
                   check=False)
    url = REPO.replace("https://", f"https://{tok}@")
    r = subprocess.run(["git", "-C", "_pushrepo", "push", url,
                        f"HEAD:{BR}"], capture_output=True, text=True)
    print(r.stdout or r.stderr or "pushed OK")